# ITMS M8: how far is validation from training? (v4 weights, v5 Step 1 thresholds)

**Nothing is trained or chosen, and no test image is scored.** D-Fire's own validation split shares videos with its training split, so validation
recall (97%) is well above test recall (89%). This notebook measures how far each validation image is from its nearest training image (dhash,
256 bits), reports recall and false alarms by distance (near-copy, probably the same video, an unseen scene), and compares those distances with
the D-Fire test images (distances only). It also saves the list of validation images that are an unseen scene, for a stricter validation set later.


In [ ]:
# ---- Settings ----
SIZE = 640
SEED = 26
NAMES = ["fire", "smoke"]        # the output classes, in this order
MAX_SIDE = 1280                  # stored copies are at most this wide
NEAR_DUP_BITS = 40               # fingerprint distance (of 256) at or under which an image is a near-copy
NEAR_HELDOUT_BITS = 40
SAME_CAMERA_BITS = 72            # a frame this close (of 256 bits) to a frame the model or an earlier test used is that camera
DUP_BITS = 8
FA_TARGET = 0.003                # false alarms on Bengaluru validation frames: at most floor(this x n) frames
MIN_RECALL_IMAGES = 60           # fewer validation images with fire or smoke (after near-copy removal) than this: stop
N_TEST3 = 250                    # NEW Bengaluru test frames (B3): never used by any version; scored once
N_V4_TEST2 = 250                 # the version 4 fresh test frames (kept out of B3)
MAX_CANDIDATES = 6000            # frames looked at to find them; fewer than N_TEST3 kept: stop, no verdict
GRID = [round(0.05 + 0.01 * i, 2) for i in range(91)]   # 0.05 ... 0.95
WORK = "/content/work_fire5"
DRIVE_DIR = "/content/drive/MyDrive/itms_fire_v5_step1"
V4_DRIVE_DIR = "/content/drive/MyDrive/itms_fire_v4"   # read only: dataset, weights, FASDD package, v4 fresh frames
UVH_REPO = "iisc-aim/UVH-26"
EXPECTED_PT_SHA256 = "bc5c70ece3e74e675d76b120243c452f8e160446698aa3bbd315c26a001ca1fe"   # v4 itms_fire_640.pt
PT_NAME = "itms_fire_640.pt"       # the version 4 weights in the v4 Drive folder
ULTRALYTICS_VERSION = "8.4.163"
FASDD_TAR = "fasdd_v3_package.tar"
ZIP_NAME = "itms_fire_v5_distance_results.zip"
# Targets (written down BEFORE the run: itms/baselines/2026-10-02_m8_fire_v5_step1/TARGETS_LOCKED.md)
TARGET_A_DFIRE_RECALL = 0.85
TARGET_C_STREET_RECALL = 0.80
TARGET_B3_MAX_SHARE = 0.01       # 250 frames: at most 2 may alarm
REBUILD_OVERRIDES = {}      # (the dry run puts small settings here)


In [ ]:
# ---- Setup and checks ----
import os, sys, json, time, shutil, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}",
                "huggingface_hub", "kagglehub"], check=True)
import torch
print("GPU:" if torch.cuda.is_available() else "No GPU (slower, still works):",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert shutil.disk_usage("/content").free / 1e9 > 15, "about 15 GB free are needed on this runtime"
from google.colab import drive
drive.mount("/content/drive")
OUT = DRIVE_DIR
os.makedirs(OUT, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
HF_TOKEN = None
try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:  # noqa: BLE001 -- anonymous access works, with lower rate limits
        pass
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):    # only needed if the dataset has to be rebuilt
        try:
            os.environ[k] = userdata.get(k)
        except Exception:  # noqa: BLE001
            pass
except Exception:  # noqa: BLE001
    pass
print("outputs ->", OUT)


In [ ]:
# ---- Helpers ----
import base64, glob, hashlib, random, re, zipfile
from collections import Counter, defaultdict
from copy import deepcopy
import numpy as np, cv2, requests

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def dhash16(img):
    """256-bit difference hash (the fingerprint the vehicle notebook uses too)."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    s = cv2.resize(g, (17, 16), interpolation=cv2.INTER_AREA)
    return np.packbits((s[:, 1:] > s[:, :-1]).flatten())

def hf_list(folder="UVH-26-Train/data"):
    """{file name: repository path} of the images under `folder` in IISc's UVH-26 dataset."""
    from huggingface_hub import HfApi
    out = {}
    for f in HfApi(token=HF_TOKEN).list_repo_tree(UVH_REPO, repo_type="dataset", path_in_repo=folder,
                                                   recursive=True):
        if f.path.lower().endswith((".png", ".jpg", ".jpeg")):
            out[os.path.basename(f.path)] = f.path
    return out

def hf_get(path, tries=10):
    """The bytes of one UVH-26 file (None if it is not there); waits out rate limits, retries errors."""
    url = f"https://huggingface.co/datasets/{UVH_REPO}/resolve/main/{path}"
    headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
    delay = 5
    for _ in range(tries):
        try:
            r = requests.get(url, headers=headers, timeout=180)
            if r.status_code == 200:
                return r.content
            if r.status_code == 404:
                return None
            if r.status_code == 429:
                time.sleep(min(int(r.headers.get("Retry-After", delay)), 300))
        except requests.RequestException:
            pass
        time.sleep(delay)
        delay = min(delay * 2, 300)
    raise RuntimeError(f"could not download {path} after {tries} tries")

def hamming_min(a, bank):
    """Smallest bit distance from each fingerprint in `a` to any in `bank` (both uint8, 32 bytes each)."""
    if len(a) == 0 or len(bank) == 0:
        return np.full(len(a), 256)
    table = np.array([bin(i).count("1") for i in range(256)], np.uint8)
    out = np.empty(len(a), np.int32)
    for i in range(0, len(a), 256):
        x = a[i:i + 256, None, :] ^ bank[None, :, :]
        out[i:i + 256] = table[x].sum(-1, dtype=np.int32).min(1)
    return out

def near_copy_groups(fps, bits):
    """Groups of images that are near-copies of one another (frames of one video), by union-find over every
    pair within `bits`: so that a split can keep each group on one side."""
    parent = list(range(len(fps)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    table = np.array([bin(i).count("1") for i in range(256)], np.uint8)
    for i in range(0, len(fps), 256):
        d = table[fps[i:i + 256, None, :] ^ fps[None, :, :]].sum(-1, dtype=np.int32)
        for a, b in zip(*np.nonzero(d <= bits)):
            a, b = find(i + int(a)), find(int(b))
            if a != b:
                parent[a] = b
    return [find(i) for i in range(len(fps))]

def split_of(path):
    parts = [p.lower() for p in re.split(r"[\\/]", path)]
    for p in reversed(parts[:-1]):
        if p in ("train", "training"):
            return "train"
        if p in ("val", "valid", "validation"):
            return "val"
        if p in ("test", "testing"):
            return "test"
    return None

def label_path_for(img_path):
    stem, _ = os.path.splitext(img_path)
    cands = [stem + ".txt"]
    parts = re.split(r"([\\/])", stem)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i].lower() == "images":
            q = parts[:]
            q[i] = "labels" if parts[i] == "images" else "Labels"
            cands.insert(0, "".join(q) + ".txt")
            break
    for c in cands:
        if os.path.exists(c):
            return c
    return None

def parse_labels(path, W, H):
    """YOLO label lines -> [(class id, x1, y1, x2, y2) in pixels], with what was dropped and why."""
    boxes, st = [], Counter()
    for line in open(path, errors="replace"):
        f = line.split()
        if not f:
            continue
        if len(f) != 5:
            st["malformed line"] += 1
            continue
        try:
            k = int(float(f[0]))
            cx, cy, w, h = (float(v) for v in f[1:])
        except ValueError:
            st["malformed line"] += 1
            continue
        if not all(np.isfinite([cx, cy, w, h])):
            st["non-finite box"] += 1
            continue
        x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H
        x2, y2 = (cx + w / 2) * W, (cy + h / 2) * H
        c = (max(0.0, x1), max(0.0, y1), min(float(W), x2), min(float(H), y2))
        if c != (x1, y1, x2, y2):
            st["box clipped to the image"] += 1
        if c[2] - c[0] < 2 or c[3] - c[1] < 2:
            st["box empty, tiny or outside the image"] += 1
            continue
        boxes.append((k, *c))
    return boxes, st

def flame_fraction(img, box):
    """Share of a box's pixels that are flame-coloured (saturated, bright, red-orange-yellow)."""
    x1, y1, x2, y2 = (int(round(v)) for v in box)
    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        return 0.0
    h, s, v = cv2.split(cv2.cvtColor(crop, cv2.COLOR_BGR2HSV))
    flame = ((h <= 35) | (h >= 170)) & (s >= 90) & (v >= 150)
    return float(flame.mean())

FP16_MAX = 65504.0

def bn_report(model):
    """BatchNorm health (the vehicle model had broken statistics; a standard model should not)."""
    bns = [m for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d)]
    rv = torch.cat([m.running_var.detach().float().flatten().cpu() for m in bns])
    rm = torch.cat([m.running_mean.detach().float().flatten().cpu() for m in bns])
    bad = ~(torch.isfinite(rv) & torch.isfinite(rm))
    return {"bn_layers": len(bns), "channels": int(rv.numel()), "nonfinite_channels": int(bad.sum()),
            "channels_over_fp16_max": int((rv[~bad] > FP16_MAX).sum()),
            "channels_at_fp16_max": int((rv[~bad] == FP16_MAX).sum())}

def letterbox_rect(img, size, stride=32):
    """The letterbox the ITMS detectors use: fit into `size`, pad only up to a multiple of `stride`."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = int(round(h * r)), int(round(w * r))
    ch, cw = nh + (size - nh) % stride, nw + (size - nw) % stride
    canvas = np.full((ch, cw, 3), 114, np.uint8)
    top, left = (ch - nh) // 2, (cw - nw) // 2
    canvas[top:top + nh, left:left + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return canvas, r, (left, top)

def to_input(img, size):
    x, r, pad = letterbox_rect(img, size)
    return np.ascontiguousarray(x[:, :, ::-1].transpose(2, 0, 1)[None], dtype=np.float32) / 255.0, r, pad

def place(src, dst):
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
    return dst

In [ ]:
# ---- The 250 Bengaluru test frames: names, checksums and fingerprints, so that none of them (or their cameras) is used ----
TEST = json.loads(r'''{"dhash16":{"101184.png":"df2ce20cf8a9790c208c546c391c330e33229a30abc8ecc46ca1e480d760d358","1100.png":"3a15d999d8a5d3997e19c8e00f293bebef8bff21fc25f8a6fca4e9c0e950f110","110224.png":"fc6ccdca6fa68fbaadd8b35eb7aeb7cd3e4d8f5dea9dd5bcf63cc6f8dcd9f8d1","110866.png":"e1d6e512e312a2b286728cd6c4e7f8b4f89ee45e7264c5968b96e3986598c5a8","116501.png":"51fc95cd34f67dd47cf46fe4b344b3c0ebd066ca56821626e728ef88ed8c4ed2","11952.png":"29c789c74986c1cce3cfe74ce18c938c338bf38ddb8cf91df915f919f81df835","124521.png":"6224fa2426b98a00cc126c42eca2e0842285b2843080308058842c80ac88dcc8","127160.png":"2bcc9fcc97e4b6e4baf056b4527049b056bc56d851c44387739d57e849c87c88","130141.png":"ff8b7601f681fac0cec3b9e0f9a0eea0cf80cb10db92fb80fb28d5c4f94ce988","130501.png":"e762e6c2ead0f145eecd4e9becd0edf4ee98e290e5d5e6f6cf966fb666264fe3","13194.png":"9b9a0bb84ab0c598968cae56ccbb4a744a744b62d96ad8b2fc39ae2cc995cb8f","136336.png":"d6c858c9796e6128e4ecb2677026459a5e1f680ef20b220b560f5596b2caa9da","136828.png":"b88cbb0c7830bf30e838ec18cc269c0f960cbe0e898d9d86c74fe3e0c1c0d904","136930.png":"d226959e798fbb87bf8ef6cceecceaccd8ccdcccbcdcbcdd3fd47a70fa72da62","138877.png":"dcbedc9ee4be54ad9a2590d83492221233123aa29a82d2c07642e632e062b971","139810.png":"79e5c8e268f9f46c7036f41ff0c7e225e131f191f586f486f4c6f4cbf8a9f830","141456.png":"e484e584e106d404c42ae6114531a716b392b5929a92d882fd2ae524e3a6cd2a","142732.png":"0bb31e134d1311d292aac6e263e2a8e3ac66c86aba2aea0a5a0279603b207b00","142934.png":"c600c4108c105a20d80098010a01aac39b809bcea784b299b0c930e7783e7a15","144217.png":"e984a9a8c991c918e2529c0999219c429c401c0ccc2d5d1dd614b658f251fb11","149105.png":"485a656ac1444bba50d27772a69296d29272be421b6216c66dc06d404d410d41","153658.png":"b0561a569906bd82be4a3e486f212f305b986b8c2b8e6fc46da5ece2edf26c70","155069.png":"f507ed47fd21fda1cfa1cb91cb9acb8bdb89db0dbb44ac86b082aa2cc60cdd0c","158321.png":"4e744c134c921c8952c5596c4b0d5b449946b123e03bc932ab44d440d4c8d2a4","158964.png":"ccb4cc94ac1fee13ee33d215e24d4699c681888384807ed07e40fc60fdc2fd84","159347.png":"fffedffef83efefefeff62ffc2fb517f567d99bff986c1debdfef9fff3feffce","159579.png":"71287528b6687358635acd2b9d2b192b3b863ac2de489ac0bec0fe60b580fdd0","161965.png":"22e13709e888ce283c69f8c8e3e2c7628e46dec6b6a253d0edc0cd80fd81eb92","164069.png":"e24c69546614634461d0506870317058702c7016e00bd004d004ca20ee00ed08","165222.png":"eb6cea24aea4d89c64c8689cdcf4d8d0c8c27cb17091b1d1e5d1bdd09448952c","171003.png":"bb2dbae136a29e02d683d689b61a76097411b801f000e400cb0cd588cb98d718","172180.png":"ce788bb0c371c7329944a3667f24df247f103f383f783f74fe64fec4fcccfc3c","173290.png":"3ce93c697d496dc94e85db815b89de8479c0f8cad8e8d960d960dd61dd42da71","174671.png":"848e85d80c6f135575261266134257895361332a72c8e2dae078c610c7014cb9","175587.png":"b492ae91af52ae3bfd2c9b97b96bb92559225f2a7d9ebef2bca3aeb13698b739","175644.png":"dc436c02ac8c346236233e99a332e3d485188ec83dc0efc0ff40fe40fe20de21","178676.png":"6e3ee0b298c7999338937a0bf811b018b0323468646c6c1b2e92aa8828484908","179501.png":"d764d360d7669362d260d7605568e4d6c89ef95ef82ecf2c6a826b1878f3f630","181392.png":"6a68284cea607380e922bc60ba18fe11bd98ae9e3b83372f97338657d456e04e","186193.png":"c712d933fb2bc98eacc3a6436b2923897edc60ce60ce58e218581c781edc9f85","187590.png":"c600c4108c32ae22d500c5144720273866703530a598e081e18170c370037a07","188930.png":"f9ffc3fff81feffee8ff42ffc25b455f002d429ce1ec819eecdee8efebbc6e5c","189268.png":"623722555167d23ad2109230a098e088b659d349c768c724a5f2a7b20a5196e9","190976.png":"b9b7b1b279d079cbe1d0fcd8cc6eed0fc60fcf029d81f9819cd0e7c0d8489a04","19657.png":"f8ea3b9abc8cb74acec31d82788233226641ee61d6609c40b840a44089c2d742","200537.png":"76556ea9eab47a3ab27df63ea61edc1ed90e992f9d2f19a65d071d171d13bed3","207589.png":"ba2a58a9386cb04890c4959673c77382d6004d005b80a18065d0e7c0eca0bdc0","207692.png":"14f892d815da26d9ce5e9e5d1d5c3d6c3a74f372e7e2e7eacde5cf65df6d9f41","210840.png":"ed26e12e145430721c2a3cab272b03a803a865a84ea04b900b3083500348134c","22297.png":"fd18fd98fb15eb07ef22ef08cf8adf86b2a6b2aad6acf6a4f798e70ce304ee14","228433.png":"b493f293ba91fc30be20be45be459e261e941f140f4a5b087f005f0afe1a7e80","230788.png":"d6d4c614c457ced4cb449654dec69cc5a9c12b813ed0b0d0bc88ce44cf86d716","234270.png":"6e3ae71297d69f833f137b477f44fca6b8323c9b7cd92cb02c98288838c8788c","235615.png":"c71989b8302c3a233b33791249145d382504618c918ab5863596d2d4da90fa92","235699.png":"de103250f280e380c82db3257e91fec8fc44db00db305b105dc9da715f125f92","236776.png":"8f368f168ec68fc79b4699c6a8c6ad0c25b44e85b486735264b1a6a4bc88b988","236977.png":"0010000020000800080010002800100005802980214010400a00040000100810","241777.png":"9b41344031b01334aa6cdee50cc07ae46c6422c43238b29a83a664a298744216","242542.png":"8a2ac91adb82db26cb078a41db61fa68e2644268bc6470b041a81bf0e7f09ff0","252924.png":"b449bb1556a196909ad841b14c61633278c2ba30be2cac41ae09c815c067d5cc","253166.png":"4dfa5f983cd872dd92d92692eca2e4f2ecd4f0a6d9aed9da9b5c9e949d309c28","25900.png":"78197c18fe12fe1af60a68086d00ee10ae01c74c2ec6af129630b37897cc9fec","262858.png":"62c6a28620276403042a20271227110d28c5048d01d506b305230c7118711925","272400.png":"b605ce05bc05bd0aa80aac84ad9175917511d4b0ec349a5caa1c883ccd3ccc3e","274086.png":"e40248884aa08f09ab0535942c55db05fb0476046506b285cd56c4c6d4e6a4e6","276372.png":"d2c4d1c4d1a6d5fc4df6b1b2ec935d917d914d9ded9cf78cff48fe40fc50d750","278075.png":"71bc752c6e2cf3acee849b8c7a90d7406d81c9c29a81da819b88fd80fd40fdc0","283004.png":"2bf04be489e40be433e43ee47ae6c8e215c6f326b146a4669ac698c6d006f286","284326.png":"2d382e172e5a2e5a2f48394a29480caa2eb2425a526ab2733a773976916a9d6c","28476.png":"2604ac00d118e1088808940e206448a256205600a6d1aef0caf440e4c610b292","294584.png":"6eb56b317974704ce09deaa6ee06ce1f890f892f992d99a79d86d517dd93bc93","295604.png":"96e170e174e86dc24d927d98cbda8d03eb95c912ab563cf23cb23a94b0908990","296389.png":"d9b5dee5dea5f4b575b19e99b9b079e0b8f0b9f03c706e646c6a3d6e3674c66c","298124.png":"e551a952ad9495b2919106743a342318299c2074cd66910e92869f06d474d5b2","308639.png":"48b76a9329166d0368856414ec86d4036c0175516569ca6cc8e4b8669a64d874","3120.png":"493341a11600e548c7481040822182a127604f490f450f6fc76ce56ce6ede46f","314181.png":"365975d865ea777e5d15588fc2e56970a130a138b48cb45eb8c7e8a74e497689","326655.png":"f8248228f804fff8c01fd208db0c5905f23468c2e48ffc3dbac8f618744de0cd","326864.png":"bceaaaef714359f9d03ed99bdb689ab8b81cf026702370112004240030209430","329152.png":"8d518f9b724dd644c642c649d6c1cf608f748c741c721a601a687a6968eb78d3","332518.png":"dc12d51791959852de2658abf118a4cc62c6c203e213e25ae24ce218e308e30c","333449.png":"0c67259428d42498288a292208826823a8a054418451a0a1e4e031b051305022","337153.png":"e8b6eb32e316e7366274637167625771d783dfa35f6b5e6ade71dcc1f985f144","354370.png":"fff6ffe2fbe0f5c0ed81ddd09d309c905990fb82d301db28de28dc285b189b1c","358588.png":"dd4150e046d846d87ee6f690e5a1dda131aad325d23dd434f408dd48c5a9ddd5","364539.png":"a864da2e4cabcb11f611949180d066d165d92d9819991b99bb1879307922fb08","372670.png":"a7989bca4f964f0667c77e737ed138d1a8918bb999b81fb9bfd99f893fa567e9","375441.png":"c8144800482048004ca136223e1a302af022e149d541f100e1a0e0623002f002","379361.png":"2f495f19b75535517d79b5adb24ce66cd782c7238f254fa9d7b08ea06ea07f88","379526.png":"e982ec4366836711d661b026606542519131906c30182450e250caca8c910cb1","380524.png":"4a69265a7524ed236c33f71aa683ec693dbef1e8d38db38eabbea39b3b499af1","381330.png":"d4249624be2c998c1ca55ce48cb4cc8e5d8255e1bb45d6a6db53b32bb101bb09","384976.png":"a66ee4dee1d2d1d5979e1736073146f906fc8af02ee827d86d8c7a387624fa60","385489.png":"ebb0e27bebd6d95cab79b9f139703d6065736993f98af9a8dcc8b19c93388b3a","387455.png":"cb619a60916025a549a4ce24cf138f3196b99695ae12a733a5a995a5b8b4d9b6","39125.png":"2c043df45cc58c639c769df6c96e919d97cdd2240e7b6a6a426c9170cb93c88b","393838.png":"54df5cab58ac5a4adda7ccb36b39859ccc98dcd2c469e4bb9158e96d6c8cc80d","394924.png":"5946334cbad9ebda52d8fe987198f3b993f0b6f0b6e0aae8b6f0bed0bec0bec0","398959.png":"d6438ec6d8cbf18bec4bed269b2bd12cd5a2d5bac5eef310f2c0e9c0efc0ffe0","402314.png":"9b1053b65ea2abb31bb4c700c710c74086400e40ce758a7caa24a8a86820ca20","402694.png":"d58dd59cd394c799c28b648d4b6aab6ab9a37c66e4849d54f554b2f4aa8caa84","404094.png":"cbe0cdf0db52ae71acc87ba1f9a4cf25cb632fb15f11bd38b470a474c9e149a1","407451.png":"dbfbffebd7d98f5ae6986de3c5fe87fe0f341ffc1ffc0fec6fee2fe777ebf7d1","42028.png":"f3923352f3954716cf13ce1cb41e780ef416d8279a277a2f123f923482370217","423075.png":"dfa27e92ee4aac4cf5c0f5e9ee67ed66df68fc32f9b2fa2af9a0f930f910f994","425771.png":"caab4dc9eb27eba6e5c8e1c8ed88c788f328bfc09fc0ff80df80dba0efe0ebc0","430269.png":"572f153b9517b6c7b287ab85899499c699a6998a998a9988b9c8bbd89bdc1bf2","432315.png":"d8cf497bed8b4e8b3e3aac598ccdd9c53285e689c799cd89cd899bc9bac9bac3","432658.png":"9c8c9cd93aec339a7c0e68904b1c4e3a1d26fab2fa20d3a6d1b6d3b2f1b4f5a0","438346.png":"0b1242b074f2f664bd10b510eb14d9003882f225e494c316c67406744471cce4","438435.png":"e698c6a0e1c8c9c8b7087701dc839d0399039a02d202dc02dc02d936e80eea0e","44015.png":"07e893f833e02b30cf7899f8d370d370cce0fdc4fdc2f1c2e9c64bc65fc4df8c","451021.png":"d6c7f187ac43b927f332f3d1f148e1ebf020f938f830d808fa04fe80fec0f700","453106.png":"4f4aace0ae68a214bb09f9c46c4066626e38db389b16f38af289c6c9eec3ec54","46025.png":"0e569c328d6331c7f347736f724a715963586698c71941b80db204b10cf05c61","466644.png":"099063b25571434063407394b354e354ee52ec64ec64d866f0f070a4e126c167","466830.png":"c20ec2a5e6e4e4ace48cc62cee2ec62ae219e218e218c608ca89cb09ceacdee2","467170.png":"5408521252966388690d430dca417a41a465cd21cc31d821fa10f298b108fe18","483736.png":"df4549e3cf318d31ab114b31c3308698c4980c882c2c340c5484961416163246","485078.png":"cd350e711cdc35e4e364c7524f481a283b99b88db0c3a45366436461f520dd88","485297.png":"e5f0ca7e92a4d1d0dba8df04e39ff14bfd23fdb6f8a4ea94f2c8fb52fb52e970","490353.png":"462bd73116c834d477457349c341d4daecc664384c38cc38981c9b8099485186","505811.png":"9eec9eac9f7c1f3c543868c08bbcce780cf8f9f8c9f8cbf2dfe69fe4ffe4ffc4","506389.png":"429892d866b976b9056964d9acdcf94cf14cd26cfb6eed6ee326bd28a928f52c","506423.png":"5cf8d708b5d0b9d0d950c5408d0af102ff00fe80fe00ff00ba40e680ece0d4e0","508636.png":"a60db60dae0dfa0eb08ca3b672e43dac792caa2cd4b87978d238c744cff0ff70","514073.png":"9a65f8e1c3e147634d794d744a746556ebd173d2f3d083d243d2c5f1c6f1efb0","514122.png":"d5a5d6e5d7e1f4f1dc55b685ec90c894d8d49cc6ecc2e8c0a83cc880e800ad04","521626.png":"648728854cc9ca2deab56e95ac635d035d69fc38dc787278b27c64f8d4c8d468","523758.png":"cd662d67898d8116c65ce13c61446b64fb15c13d9c7876d2cdb6c9b641e4536c","526014.png":"d249d3c9d6107750b1829261dc20c580cf80e780df90ff50fd10e940fa88f288","533942.png":"c119d919e599d4ddd45c575463102381a32393a193b493b08b908859cc68e628","536839.png":"ebb47fa4b3e4f2e8fae6fb949dbaed1ccd70cd64fd90ec60d8c4d100c6149090","541303.png":"96f3a5efb71aa50ba711d325cecda310a349b602d290dba2cf262e3416240e6c","54303.png":"63424b5266c6524148c05310f58ce5e1e1f0c1bcc1b6899bc454c12ae111e188","544842.png":"d31fd70ced06d625b7206e14df4abf1abe06ff43fe20f800f110f402f640e548","546564.png":"9210b134b3669742e6349c7e5974616ccad88dd23bb0f760ec60d6a8ff40dac0","546774.png":"3d6825685d490cf56decbd84f325c3290c65186de2ecc4ed8bab0b6a4a7aceda","547253.png":"7192e89a0693ce9135b1b9b0b3dab7f8c778176877ecefdcdfd4bf94df248fa6","551599.png":"7d867586b587e526c826cc64a6d256229c8bec83b852ee3431517632b5d54d93","574487.png":"167084521650128c9e042956bb201b34a6386c184e0415071c892a1b2e126634","581339.png":"f89cac99e71b459bcdb3cc53e48acc8ecb06cc4ed68af4d6fc16fbc6f356b746","581773.png":"5e297971e191c32a855bcc64c9548544a50532067a0f560e060604029403c923","587187.png":"d11bd98ad2cc3e46352139291a741a54193c1929192d5da45996b10aa303a301","589684.png":"dbb9ffebdfd90e59e6984fb3449c9cfe0ff41ffe0ff85fec6fef2ff767fbf7f1","606053.png":"2fed59ee3ce69d6bdaea9aeca7cadfe6ffc3ffc5ffe6e5c6abe4ebe4eae4ebe4","61403.png":"e504e560f680f9cac4baf630dac68cc8909880b08092c0b1cd91c599e599e11a","629029.png":"ff97f53def355a895c153a6ab93afc4bf853bc23bc25fe18fc18ed18dd067d06","637122.png":"c8bae59695369534963418342835d0edb8c4b9cafa80e4d8e5c4f1b4f394d3c5","64003.png":"6cfc6ad8691aef0caf96773bb72bf393c3126d135b19d51ad51cab19c208e320","643563.png":"5aad199c798ee3cea2c6e5eaa5d6e993c5a28598c5d9c558429ccc944d992d19","644081.png":"4a4f52ce4e8e4b03803bce225024660861c6115890dc9290b481ae4462a46a02","645326.png":"b50475467d437903f981fd90eb105b88f388f78cf708f710ef14af14cd94df94","651449.png":"0ad84510833ddce3f1c825eb2f6ab7f62fe63fd257d33fd33d983988bb9ccb0d","652367.png":"1aac9e28de2bfd63f86af923f5927f93ee28fc00fe00fc00fc00fe00fe00fe00","655387.png":"dbdadcbb3e9c3a1eb89f7e8fcf46cec86cb476d3d3d2dba2f1f2b071b069bce8","661657.png":"7ecdc4d52d4e696cb95c69ccc5cc9a522e127e92dd97eb92f5b2f38aeb32d72b","66500.png":"9e10ba8aba88a998ad94fcd478da784c7c6afc619a61df44fc10dc00de485e60","66832.png":"d952f95552425b69992b9b7bad29a9a06db81da41ce51c6d5e79d0d5c8faf7a1","668419.png":"ed2be10fe217eb0df21fe3979b9ddf0cd70cb708ac261e12c604ce0468003800","671140.png":"b278e6606140e440e6a8cb2a8b0a6fcac706c0067c825cb01c0459005d007d00","682810.png":"e803d0b3728a642164269900b98cb98cb38ce384c384cf8486e6baa48ce4ed20","68330.png":"b16c04be98b6837688358035b21ea018b01c7ade78567c17fc47ff277ea3ff22","683595.png":"452c448c468d952633b4c4c6946304319e98ae1e649e464f4ce348e5dcf2d0f0","686114.png":"80ce346a6c1a160634d23652944ea899238343826583618b658b368a32c252c2","68903.png":"99979bd7a0d5b2d9ba58aa668a4aaa72a852a473a41e921e9096da8e5c18e360","692196.png":"1b4b530a4733d732a432cc239837759761efcbe69bb3b3a94f64a644e724ce40","692303.png":"c63eef56d29cd998f999cdcc47764cb06498e318850c0cc709c733e727e14fe1","69976.png":"726472e8b8e068e070466602f802a002c901940cc100d90cf994f3d0f400f010","70239.png":"516813ac4eaedd6bd5c8d948dbf8b2ca66cac9eed145a1e18d91cba3c7a281ea","706259.png":"f0b6259e228fa50da11a20cb3464926a918e6b5526941abc8938a73027a0b3e0","706931.png":"e65899d8da9ac95a3ed1ddd45c94dd18adccade4ccf9cef98aee8af59e536e5b","712192.png":"f66cb65ce555d763da0dc288ced8cb3664764c324c326c3c2c7b1cf94c790d78","713343.png":"6c3ee1b299c7991338137a0be80df804b406340274006c022e10aa8828c84a48","715242.png":"96983614d22a950aaecb2f086f2dec6daecd73c933d97798765e7e5eff5effda","71748.png":"ce189e5b3e0dff1bce093e193c1d9e1e7e0cf84ce64cfa08f8e0b86438a43820","740400.png":"33a1768925a9ce4abce2f8b0e3eac5a6e5e6f646e6c2c9d0ddc0db80db11d913","750600.png":"de707aacb3f2a282b1ca7b027682cc26288cad2c75226a3668b6592279a4f628","754472.png":"7b246376431763317160cd0a5e467b2466ccbe70bcc4bca98f59d5ced7cec717","756528.png":"ece566c0ae82b602b0a394aa5cca3d6c9c2fbd253d110d852c86ac96c69693b4","757654.png":"710c3184f4846c84f84475c554a0d4b1961bba1bc3e3ee43aa42eb06ea06eb06","761669.png":"9254bba5f539eb65ca8a7b6fd46c846e94eec5eec7ba877437e874c8f9a4eba8","76552.png":"c1f2e1b2f9ecda794a75eb45736bf2f9eeb2cc30c838c976c826c4a6d283e908","76645.png":"12349235a55438784a6451ccd90d20cf7e5881411519a59c5dbc34fc6df407cc","773162.png":"c598c894ca32cab4c8a5cb145c10a4507498584859945b064e589858287039c1","773806.png":"9d464dcf6c4f0c528cde3c675d15bc25b926f44e74cc94ccf41c549c5c965cb6","774596.png":"fb859565bc455969d069dc6bcc4ba72b908bd162353d1519a31a83358b1d4b7d","781746.png":"eb06a180d999e3a4c6d6c693c6a7868502a312ab14a509a51da02daa2da23da0","785007.png":"1d58598a4dc87f2c6fa47de665e672f333eb4b6c7874b364b7b6cdd9eeda956c","785732.png":"e308eb8972a033a47391e131eb4cec9ece1ececb909da6f866592b390ac1138b","789065.png":"3aaf3b473ce13534b39a32cc78e93965d971cbb1599851d94ec8fcccbdc17d88","794461.png":"47747b701eb4d4991944736675e2f5a2d661c663a661c6738f626cf02b60ac64","795287.png":"58db6ab9ab39b819b458b8487a4e734c7b287b96ea70d360eb11db018b20cf80","80005.png":"dcbcdcd6e4f674a5d86599f0acd22a522352658260d4e2c868e47cac2532a671","802597.png":"b3a63389a3ccc3f0c3f889bc89bc8b9e139f111d990d998ef9a6f988f5b2fa90","810331.png":"f82cc7c8f806eff8e80ff21eda0cde45d51ceb64fb64e3a084b49c0aaa9caa88","818164.png":"ee3ad62af5b8dd3dcd136d73ed51eddf58dc5b1e5b4b330d9699c318c93af51c","818557.png":"c2b0fec07e48ef22e371d3502b54b690b39013a01f023f20f3a4ed219c22e3a2","824427.png":"f349f249ef1bcc3dcca9d9a9f129f151e151c149c36523512d109f40bcf07850","827304.png":"e0ccf038f650d748d3b87778a879c138d0fa487f081f9a1f1827310592081180","829240.png":"dab6ceb6e8b862dc639475dce5dca78adf4d950db70436847f4e4a8ece0ccf0a","829959.png":"ca2ade12d3a287047b623a52aa226f13ceaecb96a64414684c74316630b1ac78","829978.png":"79d16c946d9c3118233ab9ae3ce658d4b974bb6cef0ccf2cc3b8ca1ac61accf2","83149.png":"d6e48e949ad5bf31d9d9cb99da3431b5b1b7b4a32c496029d261801be01aea38","84796.png":"df3bdb8ccf0afba1eec85eca5fcadfc4fee5bd63377abf309d549980b130e128","848486.png":"f3357354f25cf294f6506c70e6a0ce22dab2e90ca92cfb04ca04ea16e85a991b","85483.png":"71b87528572c1f281c2276b0f6b2e6baf5abe4ebe08bea6be06ba169e271f1c6","858005.png":"81f692ce72de72dbd356946355c727a72f6646fe41fee3fc4c3c4818fa39f879","858355.png":"1731057647d217c6d6504198f894f53ef627f869bae3ed60fda4d728e739cfba","859158.png":"5a6669119023995727c31fc9d34d4501ed895d955a61dc39bd637f52fd52f09e","861398.png":"60ed602d206c580b282f4812ac1e240e861284c38ed38191931d91ddc8bcca58","86218.png":"b58c82acd24e58f2f340e301ea01e004e004d00c8218c830e270c4e0d3e0f7e4","864056.png":"719a58e2350c350972e36279532ce10a990fc90e90e7b461a421e43ae45ad633","86480.png":"895ee19e313a33337f176db56fb9cad19a69dcada0cca453d4646c057610cb3b","866078.png":"5dc7f5a7d6c3f9d6f34af3c1f152e591f390f918fc10d890fe00fd60fbc0ff40","875190.png":"95359575d4537c38ba1eb9257b0d7b4d7b2dbb2d3b3d791d5917714623626370","881888.png":"2c6834ac3424383422b26c664933d8c4c0e748110c19440d441f441de43dd23f","893031.png":"471d471a5b1a7a4a66a3ef8083082e00bbe093cc04080920923084108250c450","89437.png":"a7a1be9df28686341c44f9ace126c7668ed69ea2f4d2f3c0eb82c583f792ab5a","911150.png":"0f309e8186e80e688e784a4c5e696b5936182709430a5f065d06ce00ee806420","912687.png":"7d8e75863586e526c926cc64a66256021e03f843b102b186d681c52a166b5e22","913429.png":"c854cb498b4c3bc73bc63ec6e8c170d0f1cce104c1649326d3a3e185eb11f111","91587.png":"2538e4104370cb70393419761b464bc88262aac2acd3b0a2e422e120c124c272","916637.png":"c608c4108c141a28d900b904bdc0bcc09ce0c222c222d2d8d270523c5e0e5a17","919010.png":"3a30395431dc7309b879bc31dc69d9797993b9e3e3f1f7e3e7c3ffc37f027f02","929272.png":"97d1973986e107a027227702da80e6e0f2c4fac4fcc8c19ac7d0e7b4bfadcf68","929878.png":"35d0369e6d936dd8f91aed2eedb1cb2ece13cf28c76a464c8e04cf844f441bd4","929978.png":"195118d51967182719d31c721c131cacc93f6a7e333ab3ea87a0cf9cdb0c5b36","931597.png":"9b768f968ed6afd68b5693d690c6b3cf139ed5eb97b78ffda5e4e5acf37cf239","933281.png":"bb2dbae5b2209f32db8bd489370876192c19b919f208e4c8cd88d988d398d718","934870.png":"8b619dc1cff1bea8ea68366836689ae86cb2cc9658381a081f489cc85f086fc8","935810.png":"cce0aca4ad66af4caf08c091a999589ab0d95899d35a4bac0386c682ce0dc72a","938125.png":"8f768f168ed687c6875615d60cd6434a58ba5b9554463ca2789172847224f369","938473.png":"712865286f48fb28a59a9dd23383ebc16fc06bca6bcace9aded89c8cbde6fdd0","941351.png":"68686861e96ba2c3a518484cc2c53287430d6343420a73818290ca10504114a1","942845.png":"ffe0ffe8ffec73c47fc47fe843acdfac7dc5fbe1fe60ee63f463dc63dae5dae4","94361.png":"d135dadd5a5b3ad9ba757e71f031e130d071e4e1c6f2fbe4b5e557d05fb057a0","944502.png":"792877287368f728cc8a2f892f9137d137d9cbc9bfc3f6a1dec8bd80bdc0fcd0","945271.png":"722006604e68996e352ca926d9cb91c836486f2d4b943370d238f018f348fbae","948729.png":"370e67489a829d109d181d0415041582598459a569a0e9a0d9a1c99059d058d0","962847.png":"23b769f6f9fbd0fbd6d8b27ab47e35d72547c5a760f23d71339130f730f6b474","964462.png":"d30aca2365b25d45764c76ac2966bd76ad6734a733893e491a029a4a9a418a41","966551.png":"96b3b577b795a58187e146f1432a3de72b9152730e3926183e2c1e368f378f77","966699.png":"6ce217982d516670367295631655c4c145845b112d899b648a55bac96cc86de0","96883.png":"ea8afb3aad5a2d6a18e818f08af09ef29efb97ed57eddeef9dd79ff32c671b36","972582.png":"d59ad278953849f03bf447d4f56c8d7caba4257656ccdb04d0e6e3c6cd665acc","982198.png":"c948ca4cd5549647b1307664f09cb64fa4c7c86ccb18cd0ce176f332f2bcf138","98292.png":"dcb6d5c3eba1fe49d72ab553ab526833be2276626864ccc0d8c0d390635027a0","99206.png":"fd98fb88f5d8fd45eb72efbace6ade67be65ba6ade65d87cd86cd8acde24de44","997377.png":"cca80a8a1e0ab80ce082e8abc9629940136009784cd00cb086108911a09a6cca"},"sha256":{"101184.png":"77d47f52c15811c69f77a4544fa409c7cbc60869465a8857c1390b01005d2e94","1100.png":"fac27debf215b5d262f096c9d098f68b557ddc8297b60470e283f1874d05fc0d","110224.png":"f92999a0531708b7a6bf918265f13657c099af0808d34c3a6db6c0bdcd4da9c9","110866.png":"70ba7808e84f4937af103dae18368edcc2ba4ffaac61704e412374a61e054329","116501.png":"a9936dd14367689e2360bc1a3769a614c1fe4fa8773dee73283c857be505a9e6","11952.png":"eb9f01e0dd6f74a9181b3b29251bb0bced3bcd143832b75640d81003ef26455a","124521.png":"29f886c54fb11d8a8e612ffbf18bd493ff6c1b9812597eace12153d4a5e43a40","127160.png":"54b36051b5bc7bae3294d12c718eee0595608b8caaca8d2f9438f3fe72153308","130141.png":"c62991daac6b03f4d6948f187b43e4ceda5919c98268a20658f8479b12a03d94","130501.png":"52b1e226d7dca22560f8fe8be27ccd099693faea9d48b72ba7d2d52c27ba9177","13194.png":"c3e8f44b219ade4e5a813748c27107ad7cb295d8309e095c098bf7074a333849","136336.png":"cc0623e5ea15a61ed21ddecfd9951638f21906ded7d3694c99ac19b937d300c3","136828.png":"500c9b7d903c61c0d0a115ca7d0bbfea3ef9a205ed11b44f9af80e1ec639443c","136930.png":"d22cb424cb64b465b7866d406a08d88b460491fd1637c23f3cb982d9caaa8c1d","138877.png":"4c47915b286296262f59adbd1eece1401ee8801d83e7cd47b875d0772028552c","139810.png":"c56e382ed16b749d38b2bc077500f6dc2ba723c692738162b028529654382916","141456.png":"e22d7b7115452e6b5f800c928cd1d5090ee23c0f14004bc432db33075a39f16e","142732.png":"4eaebf8010deb4b4aced3116219d5096513870fad4b2c366d6dd54e3c27fc64b","142934.png":"0d43b9b9b4ba520c079aa0f980a7cb838168083240761550c8a29659f72a4ba9","144217.png":"2c49ca9e8073dffd0b9335e4c1f798b69f673f088f4a2d6da93f6de7a769b0a4","149105.png":"36572a7ddbbdac275f9e0e0f1acb9358288d1b4a42fc3209aeac25d68266ca0d","153658.png":"4f0000f25faa4b10540b9c829482dadfd276c47e68a05cf9e89c48edf8ec289b","155069.png":"19f1c99b291a80e6fe6d9cbd2a4f4e45cefeaf8d2d10cf6b14c8abe93856b90a","158321.png":"5a67565ad4b68dd951a4884b9110b8fac6f272d97b2b0df98bea8221a55e61d0","158964.png":"9b6caf9ef87278f447eec52dd969be257755bc6ed8434ac738ea8b8cdf89dc8d","159347.png":"23c24ee3a61cad869e61659cc5e95a682486be3e8e70c36bfe8edbeb7291cc7a","159579.png":"f3c31f57520107b6cc428c89f3cdba8e88c451e0f7a8f9a65565e5aa724c51f5","161965.png":"490267c34f6d09bbdfa9799e2674e61a4a6b3c27d7bee5aba775fe3383312dec","164069.png":"37433ad0e39cfae7a272eafc7fc02549dc5990122729caa36d77e08edae9f091","165222.png":"70785e8242d4554937d5908df7d0936512ebb9afcd8bb964f11205b8bdc4d1c9","171003.png":"7113454af595c752fbe652aad283e6a2bf0810390367b047e08afcaa7a405656","172180.png":"a4f3b991260c1b00db6c42b7d6587d998cfa3acf06415e48c12efad7ab776828","173290.png":"b8ca0cf05a7ff588f90c4560bcca1714eb99f50c4bbb2d69c1ff24dfa81d19f0","174671.png":"c486a52f8dc39d07f5bcd4851335750d13af6fbffdc953fa94f56fae38eea30b","175587.png":"1f1ca4dbcc83ae60e35de5f2528c7d4d827ddd3c39d591f6fad53acda9aa8a11","175644.png":"a2aa886b5ccbdff7f33fcb3c9429e9f02c018b1fbf19e81f2dff41bfe0b77ac4","178676.png":"1188def8433714ce7db0643d911d9f1de3e61edc562403efdb9105ffc9e3ff08","179501.png":"95d3a1cfb30e2943c3b0b13383cd9351c6b444671edb21d55abc951413f15e58","181392.png":"ae9da9f13c93ea708bfdeabca1ee6feceb309d3b9480d0ba2d94f7dbd54892a9","186193.png":"e48d419663eb83c4c15cd61f0dcdbb1fd6c451b5b062dbf10d3074dc59bc0f11","187590.png":"16868212ab77b1e2477da9f8aaf9844c2fac2776ec2389adc9fbfa3eee3b7d89","188930.png":"258cf534f8be0212260f8880c03e5acd32bf97b11d106daa87388fc00753e438","189268.png":"f77aab386a73836091dddcfee808bc8e7d4cb884b9c417447714949c87aca058","190976.png":"146ea316846891343e2bee55b53f472e8e8422cf9a3de65e73c181b798afbd17","19657.png":"5844e5e9753271ff1ebb1db9bd39c2c58498ed418c16f88aca933019ca5444b7","200537.png":"7a7aaf777ca7d3795b493df1cc51751ef922b2cdaf495c93b3c79056122b5ae8","207589.png":"826c5cdabbafa557eb51d4870f99da76fd6cce31d7e9e0b907aa5fe2bcc49186","207692.png":"0bcf4e17d6d9e0549dd6f087cf154fb3f6c9a19db484121056314d46bd5c7382","210840.png":"cc3180e2bb28268376dc4bf1ec137353c6c3385891a9c0ea20e2e57e7a3ab6fd","22297.png":"3789e904a0bf040cdeab168014f02cdd04510b379656f36b0e70c444d4204b1e","228433.png":"a10ef320ea002f17491469a64e96e7575ff2679f326aebf17bb681245f4782d9","230788.png":"d7895fbcaa4d88fcfcb35cc0a379ab001dd1600ae79c3cdec6a481380b0d4869","234270.png":"da679f70e532bc078614a3b815e681f9facfdfd16d1e04e249b3ae59bced928d","235615.png":"1443dd2581f7bd1a766c2d0dfa713201b99a278f493f663ff0aff7757c3e70b1","235699.png":"7e8dabe3e8a2b50911633f5973e8d0be0474d5d76abff6d40d26ca525dbf072f","236776.png":"6b46dd0103a95ab8d374f53d9a790a1f3bc3f70444f29175ca80c1e4eb5d7ccb","236977.png":"a20677e2e8f42d3c13bec8672c2a3831a5126e626ee22eb6740753405ed57d12","241777.png":"959fb0b8591ec72e71201de6ec9f960567e11db2f61fc5e96c5e31a9cf4afe7d","242542.png":"8700c256960b12c3e404a92d23089f75fdecb0caa917e180d357e3f524254c40","252924.png":"221dbfd516eecc2b2d6534c9a88b11c6249bf8a9e257ae1b62f9fe74ccb2e72d","253166.png":"6124b87540049f3cfb044526f3eba3856caa312ca58fb59e97104882c3b3c9b8","25900.png":"e192a4d6ded86cc32c34b03242577678c3e4f67b3d73acaec9f79a48df5fa384","262858.png":"d70096788ab7bd12a98d2285b28c371751d12825ad4ce55de91ee0700da46b10","272400.png":"3c0739fe7a96bef1dcf89a417d3502a08033f226a4a81c48c34f78094e2182f4","274086.png":"5c0dfd8baa65611ff5f4b0e606bf4d2c5e1e3bfd490c988024b79db38f872f2d","276372.png":"ae5e38e6ca7b579d91b9cb0fe9396f528444592fa4084249163ab01e9f5833ef","278075.png":"a0f7a5abe5ab01e2092163ebb084c35e523a5a5dc7689a245f6536fb85e138bb","283004.png":"5cd77d5921884d8753623e204b33df4b91da6496c2147354f7cdbe6141e45fbf","284326.png":"446989e39fbbb704b0a3dd89c705ac982fcc6831320d21f1219e9c1281592edc","28476.png":"b89a4ac0853dea03e63eec150a2f4b07e914d63a30b75785913f192d586318a7","294584.png":"2697f5c2cca90f84a6ecbd08675d12fae326dc4c06b1a6af6a002fce620e4199","295604.png":"2973e9d133cf6551e9c9d92e7dc49011cbe0190dcdb81c2950890885f21a9ed1","296389.png":"63444bf736e5f80a5922231b6a7da46bc1c796c3172bf3d51ad4b5714528d975","298124.png":"c80870eb7c7aedcc729144443c1df5de1ecd731a3bb7768326824fdfd39eb561","308639.png":"0c40f1a4c88e609629c0ffe0f8cf06ef05e88bff693591df0ca279459577e3f1","3120.png":"0f9aec468a3a4b15efa9de91b85c8a26a4ef96f4f5de44e87f0830e42a31c315","314181.png":"42154477508337cb2dd4ccb97a3f9eac98781c2096b7809ca9a81ebb1fe36854","326655.png":"ae32c8c79af8313e5531f39e7fa1e484571320c10c97f2e5d0e7be9c72307476","326864.png":"5d66cf1e6bc2a570fd23e7cffed2b4e9ac8e50a43fd0036a5790ba6de6e440fc","329152.png":"34634556d04cc792881161bc07ad4d19cce15612d3e02d860f4dade5c45b2026","332518.png":"ee54f4b10dc5f3283caa1fe89647f4275c3c3e1c2bce4c8d6ef2719fa5d4945d","333449.png":"1c419e07b37716d23cf32f67724dc5c8cde3020225e096513922709993740211","337153.png":"5b91f39ff4751605ba9a65342ae617a178cd4aa153f935b105fdf0a5a14fd30d","354370.png":"d2c2407747e0b5210369184008db1bba758225724c7c9cda5e29759f72b912c3","358588.png":"c055dd590579a862aef199de555bf69069f984bd7e44b7a9311c6396bd6dbf9e","364539.png":"838ef4b09cadf7f83f3a1bb8495adb98a9427d31d576c44f33b7dd7b3ffa8a60","372670.png":"c3748540348faf017cadcd2c7a10811eb5021ea1bfaae7fd6c7f81a34f842aee","375441.png":"d3bbfd847aca0ad537680e05642c615aa55431fa55756ae269b27882a5c64c94","379361.png":"2e3abfbea8c239e737cb0b4f63a29eef2eedd897bc5685f962422ba90808a267","379526.png":"41023aae0391d0e19b875a2c99357f792c7cf2c68aa6554c863a4a9b0861ef25","380524.png":"bb9cb81f0df4e696986ddec4a6d270d654a1ff4270f24acf51d40b54925870e9","381330.png":"67b3abe45e5293235f303092a36800c6ed81e041ff877befbb6ba4d55057d449","384976.png":"1d1cb3f4a6bc2a68fa8c3f7cb5c0dc8d68e157bc813a4dec164293c5891b03d3","385489.png":"353db52ef1cf59fc30b9872bd0b17973e65725b92ba14f87d257c2334aee3307","387455.png":"d4369eb8a71f9e6652d62640bbacfefc7bd658b0314c62bd466f931eb96f6a23","39125.png":"c373c6b524f6abc56c504d922e6bbde8727db1873c2c19a4a5834a892d50e031","393838.png":"49b5a6a510466506a9525f41878515ad879dd5788916fef266b6a74a00f29c26","394924.png":"bc028f78ad273f0acb876b5468de2d24cc768283ee1aa1c2278de8f21b08b4fe","398959.png":"0a77fa5be07a8a9aa35d7d4c94af8132ea5c5b25d4cb125592df3904c3136941","402314.png":"8c983362a41b9837785292b6beb71fb27ef95336aed5641eb728f373c1353ce6","402694.png":"1057f38e05daca33c25d9aedd560af02a66c08479b3453cf0c0fad73f8f24161","404094.png":"183d78407c5cef6462805cdc6a915179395282a09c44b3f7a1a21b68ef698d52","407451.png":"b854efc0b3d323631f6075eec15d14e251002712701ee5566145ac45e10e181c","42028.png":"046d21d15c5f3b8a130112077f12635b65647beb0358122632401e9232d7447f","423075.png":"bc5d9301c039ab7e8acda6c6f72d5e0ecc1ec6625e3b7654e0d8134d945da40c","425771.png":"324d2b76e3af50c81a4097d7bec1c15d4a7c7bd99bd6acec0d2aa5840906dc65","430269.png":"fb9b61636dc18ec690854f0f9afb8a5dd0ba87d901dd4ba135b3fe05442fdb81","432315.png":"db3bc2926b8bd378d6024fdba7f15f82e28028524e7a32d621915111a5b7a282","432658.png":"24ba09f7d71d8b9bee7a14922b71a4968704cf7a3a7afaf8a017e67692222bd7","438346.png":"ae7dbbb985e78e573e75cb136557b038f7a6e855bc3b9f60edcd7f092b24e761","438435.png":"77ab25196cbc42ed54a3dd7ad13a5a2ce2ba1131874235d403f3947ce27ad7e2","44015.png":"308c66b290406c02fb55f7f3fea9a73a16c04024a984547ee26652087f915392","451021.png":"32ae6a9d5b9ce0133ccb4a69e57a69f8a6249469653cf8a06e2a9a6027c2a239","453106.png":"750044d3378dc2ee24846fab5c404dd4998440385dde03516af50d2c9082a559","46025.png":"bf8f6fad31a4a3b32d75928f49b1fa42667eec508563b3530639ea27e0d7fe74","466644.png":"45d0b60d2e47e4427d40b272e79d8e943450e7a806f0b55393c5fb9ad385cd4b","466830.png":"eba7711a56086ef66879ec1c3e0107b13368a85225f39b79abea1dbf88532c13","467170.png":"6a92d4906c87d0b7f7b03180c3dedd76e6cf36040aa51d3a776299c821e899e2","483736.png":"07107cb6c96b684f1639010cb3f12cb9e2e60562681bfdf9bcbdacbd1b1f40b0","485078.png":"62ebf9f74751076d7deb9818190b8b1245d5233d1e6e074bb06f8cd5143ecfb2","485297.png":"2018ca35fd55cacd2025828981d59fdd55a210d287be2b3ca4c46961358b8436","490353.png":"f1a00ec6b334c18ddf523eeab277f123025ea9372b225e90c4902e622c88d4b2","505811.png":"41b432f49eec57227c5f6e098014b6db5f86f70bb0215a86e982cf7146b44a85","506389.png":"dd23c6aa405c57d3f0d728a0ccdcdd82bf6b57f2fa4c3689ba4e02c8d77bc7e4","506423.png":"b59eff046fc06e4e14b4f7d950eec2ff5fb3e38beb6b51ec6f6c0496228eeadc","508636.png":"75bf19db355476b376ce272eeb047a7904e507fe526d72a7c6d5ff663d95d004","514073.png":"71b5953bac716cd789e83ab0c1b65dd18abba55e8b137eaa72df475ee8662666","514122.png":"dd8ebfa311d8f5ad6fc3c26e74c1f37ff1513f7b4783cb9c2c78363654cac3bc","521626.png":"66d646c53cf70bd364e1ab9d7fef9984d0a9c09b963a57fb4058e48294caa798","523758.png":"b274546c1b6ecf968eb00e86a4a07ef548589c06226e6f22c36951998917c027","526014.png":"6d887b373552a152e54d65a75ecdd0165c3364b9588aad6870f4ded1021cad7c","533942.png":"087bd887d7eb40cea6b3e4231ba71cc206e260a5550999a2e16c088d38d14de4","536839.png":"1054e7837297ce0e9dd08a96186ef2206ae441beaf587d285e7d9c709f72773e","541303.png":"e98f2a56a030f12cb844f47a2082412bef97d0f31abb693fdfbd00501facb20c","54303.png":"448908c9c7efd5f92e1e59c337a1cb6703f10a196f17813a6c2e10168d2fd34e","544842.png":"17fce385fcce3a0d985f2bc493729722f3a14764f3ad88b464a6f21f1e1c5fed","546564.png":"78f1b315b35f50a39753633df547ca8aa7e7f8c98edd01bee7c326219736e701","546774.png":"bb67a4f50c2b0b68de398a8b32a5bba385ee621d4e96d2faf93c9049a8299330","547253.png":"4cae467ab7e627d80a7dc93919ad752af964f6827e688ebc5df21c0d610d5114","551599.png":"7c192c30eb9f3b8789976ed082d9cf7c4dff9a4a5a4efc19fa413cb6904865bf","574487.png":"80d3b138d9d203b85c1708de987a6b6cf2265b521fd85a716cc105d3e261289e","581339.png":"70a78e0837049ca3bddef0d8dfc90abdc177c2c7a0140dfab93521f2e86cab4f","581773.png":"69395cb8c0840e575e1321d60aa1c9e8b4de7804900edda1b298d458133fc581","587187.png":"3998404f1ca68e7fd00977c8e8a5589745ce3183e451b5ceac2175583acc1bf8","589684.png":"b1bf88770fc5ded8e4fec66c0ce68ef167d8cf086fb8e6756d509a9621de88be","606053.png":"1a8c9da7b5f75b4ab30ae89c020746e3ee770d0a97d55e0209c0efd101255b79","61403.png":"0862822c62b3672c8b0779ba033fc4e19b8f4a747fb68ad5cc6032aece1487b2","629029.png":"0e3c6313af16ba5cb08ec5b4ea834a15f6040b3562ddf5d4d2d7bcdb6bb7ce8a","637122.png":"1e6ce4a75177ad436508815d723eaaeab252b1b4a47f7957dbae2c15e945932d","64003.png":"e2ab3f244b053115de9a4e84c6e56a19fd0c040a5a1938abafbc52cef39f71ec","643563.png":"05c4527ba47da0c2770ba06f1629f43e9f872535bc720e52f0f6f92854639e63","644081.png":"d8e3da5efbad1ca0cebdb12d3c74e0fa9fef95a050074817454e0f198e8c57f8","645326.png":"1537393441937ebcb64bb92289ce1fc92d362e6487b70478a2fa77141c5707a4","651449.png":"134074976770943b24ef4dd0ccb333e235fd5e631d352286c0d2839d8e45ad74","652367.png":"9d95742a641dedb0580b45a8551fecaab44a3a93e485ae748b67a9ffc5092cec","655387.png":"cd573996506e6ef77118e1fd3aba0b90577994b54231826eadedb2fe0918b7a4","661657.png":"391e53f78f60eacc536cc106789f494a08052f34c32f59fd1089c9530c7f0003","66500.png":"0a9095b682e244792de6a50cdf3d1461d4d97bf077a10024530d6af23af169ec","66832.png":"5f59f37df070f52d14b9429853863d3313da08d9d11f96dc1d5e977322936b3f","668419.png":"aaa9583412cfa9e1eeb615968acf7b6dd37ca3d1acdecb2deafb4ede5f984a61","671140.png":"1b2b18fb48c0098aee5eeed80df497a0fff558e88ef3864be9cb51f244eeb9bf","682810.png":"e97bda7854e5ec0511d32ab225f7df566e54587e3c07ea6b0de623e16f12ab4e","68330.png":"f8c1c70e7856bcb78f47ca999c3b72a34ef67d1c00bd8259c0051f7c35672ec9","683595.png":"37e6acd257ace915743b4deac3b3bc4bc85eee270d5db96caeec07a6f1ff8e3b","686114.png":"7a06a70a5593ffccdd0fa19ab19b02b3475fb0c46b2785d1e629b5c0b1b8b510","68903.png":"ad4c4b64bfdf524eb520ed7d89f57e68b01c3c8af4abe7ede6b720a16780e89f","692196.png":"98da63254cc7106f82a61952ab7a2f654d23bfeddff82aaf4220ff210466132d","692303.png":"56b10832c6a6f9abc55d2b957dc40b0194fee9bbc75754d6671127362dabe155","69976.png":"9f900dff959afe24a97e89b64c2a6cdccddb4f37678ad9ca32720814adf71cbb","70239.png":"2624f4e31172dc7d1fb2f792426fe2d1271af09dbfc689f9f315adf0be3761ac","706259.png":"7e9357b6f4148cf5c94a3d30a371aa0c2a77bb1fc6e7a5e17365bf80a530f660","706931.png":"a2075009cae2fa904fb8753b6488bad7b739ab69ca8deb4fe9b0995d3ab691b5","712192.png":"e1c8452398829ee2efaca2220fca68762010db45d6b5492fc03dea84e8a3b6f1","713343.png":"fa48176851d25cac08ad2f2964c8355f2f05402583543c4af8a54410c2a24356","715242.png":"58877435a8e04893694fc71dcb115da62791a7487c6e1c9f4b6f145bd4769a87","71748.png":"6b2f48c3537fb85591ca124656a645da4bb183f2ae6a7b59ef1c20d6e9d7b5a3","740400.png":"4b73501685a7ac553dd927de36b2c0a2b14e5268138f7c4e4f87e08f651f9c16","750600.png":"b7107cc5b37b2647e73c4a82cb2ff5a8d22585555b9c9a241e5dcface1baf971","754472.png":"10d9d9e2ea6e98af52b4ea3e75b1fa82d5f9b815c7265f6463c9bbcbe80c8c75","756528.png":"712d72929be4a6fb494ad10fd3997b56e216cae1fe21723c8343fb19757b8c3e","757654.png":"5e347a388d04ec3913cdde3a74a3c52ddac1988757a71cdcf8392bbcb0154c61","761669.png":"de9c059f060b31555d9163542af3baa73a7b2d9ddff4fb2c883cc9cae56c754b","76552.png":"9e8be722fc32c09af0de2e82068a223aca506896036bcce0f9e8635a3029fd26","76645.png":"6f422429d6fe806eaedbc2bb90a872136e68dcbbf7a09f20b546fdc0280032a9","773162.png":"8fc7628db22d295679134566a8e70055918d7454e36c0019b1a74d75bc704a7d","773806.png":"ee13c8d14ebaa68c95d2b5f93fe2c422276e61edd35677f6564c9f585e4774e1","774596.png":"c9213bebf9ba1ea4476d9744cddb5fc1c65ae898285e4a530a2d06e2c1ca2899","781746.png":"867bd36f36fa6fc0fff9d5bfce03ae1db4d57b89f08b6e9c98515e0d60f30497","785007.png":"84c2bd8fb4dba5baeee830bf84894b3d0ab9bcec3f241ea9c25bc65c1f75a125","785732.png":"3f9752c8d4abbfe7def53f2e9ab06dc6daab659e58a6f1b4318abba4dddfb496","789065.png":"a389eac3a150f15963afd2b8b8759a11f4d434544b96c8699561ca3609596698","794461.png":"cdcab6d7bcdefa23de4f689fc621d4e13e9583bf7dc1cb1815d32bd0724f7510","795287.png":"94f5f32cc3669967bc4558339bc06dd9f0a3ffd1b69371c6061cdca569a9afeb","80005.png":"e2da9e547873e87e59136ec193eed7f865585625c7082741fa68095eb682bac5","802597.png":"1ed7b896a6d55ad09a0d4d07ad74af5008512fd33f6405a1d31a1d591c14b9f7","810331.png":"e1cb220074769429052472225c45bcc8ff665c7d3a7ee18ed723fd72bd4dd05b","818164.png":"975e9346f5cba67995365568616daca377e5355c76dd1815dc9fad419bdcc135","818557.png":"2b259fdfa5003c7848e77eb42dd2fa7f0d52f28048ae18af6a7b0da5b22570a0","824427.png":"aa5a8b8cbf277603463b1aee9f2ec052d9b9cfae7959229813a8a56ad984a28c","827304.png":"ba414f9e636fb915eb953a74d8ab47fae723b63af6082ddafaebb8ac4b21c8d8","829240.png":"d84bcdde0864e2c1ae9625913189d4c82055a5dd9ac9094360fa4d5a20162a40","829959.png":"ef60ccb0e39f7d35a5ebe6680834f24d1f6e2e4f87d086a522128be935ca2525","829978.png":"d4e8c5349b69bb3437cf477371ccc0f5274ac55bde769112f5c174b760bbc992","83149.png":"48db558443cdda1b161fc2677de4c18d7d49ef68719799d9da4be507fd77a8ba","84796.png":"6f688accc4c8b7319e17b3226262151bf92daa785757634ca37042369e3202f8","848486.png":"468bf964904d12142d1507760c83b11a6c502fc55084b9309cb35dae697082f8","85483.png":"50f471bcc5758da1631c7683bc85231f68aa258917a5ced12da1b3a6094e478f","858005.png":"255bb8726824066caf13900ec7e86cd236d273e4f83332e3a4def85323472325","858355.png":"37723422a126977af142b04805fe46ca18e362c9b3d5e63dd3554cc53523cde9","859158.png":"852c3a77bbfa5eb707cfa283afb34cdd11e23dcaffacbeb4cb6d41771333fcf7","861398.png":"edc1f5c2b8ee67cdb22468b9409e034f1760b6b86d7785017a91f02a409bd02f","86218.png":"9ffdb4662b3026b00d831f56d89ee609d8c6f0002e5bd1780f581dfdfd3ead45","864056.png":"d28bbab6f9d1a074a1165ab899c251a684ebcd7703ae42babb935f7eac8a5b58","86480.png":"3e362f44bd0b194ce98ff98266b21481daed3d8ac8921423c2776f04357e4167","866078.png":"435b82651e8a4237c6f7062c92d16dec6b22dc3b5acf1bfdc06c2913bc1a7bf2","875190.png":"bc897c07be9c283c4565d47ed6708702bcc9dd3afb334137d94057639193d356","881888.png":"a8e8ae3b622d02419c9dfaea8a8b6b77e1dbf1ccb55c152e79bb7876d054d231","893031.png":"ed352be63d9949295535908a696e4a79f16b7fb2d6a5c741b0c7e9155d1e3260","89437.png":"173376a275e450b1dcc1c2932210ebf155539d6e23919ac2e6cc711adbfb6361","911150.png":"3fb63dcd25fb9d23e0ebec4eb2a770f1fd0c50090c1ad148837fc4e8b9ef7fb8","912687.png":"c51d53197cd644d52dd8c05c09e25c29a70bc96bb5b252664806fd09b74c338b","913429.png":"97673ae8a3ba82fa5e4c636e69a3855258b640e8cccf493ced96cd0f985da257","91587.png":"69d1da9008ad15517b1fafdbfe350dcb5c338be7f8bf3afda3caac79c10a2a5b","916637.png":"e432af0b718cbd7745b244521427f7f48cf1823f87e8180986437a5c58f02216","919010.png":"e6d240c91e3e1e2eadd508911fb2c74689d8462661e0d61b54e1f21f0cdc72a1","929272.png":"cec6b87b7ce7a427fcde83ba1dccb2d046a419b927b0ed9158c772b3e142e855","929878.png":"0f4e96d2c90b30d3b50193631ef75144736eda76d7dd68c5bbd3b709dbbde7de","929978.png":"2b9c79ce5756551e7996bf7b5eb1f12868241cd6bcfd2827687225cde59802fa","931597.png":"4ee99379e02e35b3144dbfce33158fda9463ccf2bf25d7392b64397357fb0b5f","933281.png":"2f60ab3b534dd46618430c156662854056aed0112020d7689bbc04f342a8af40","934870.png":"406c2fb4b03d1ffcb5f0f4286eb2f2afa05c8318188722440b38430d491143ef","935810.png":"dac31b5c5644b4bfd19e3b7de01db802a4440d1e08d049865f251fcc7e0660cb","938125.png":"e1ad26b7e5c5be1ad9e263acf0f5fa5a62ab40197d0c4c40722c422a7f583797","938473.png":"3ff79f682c1612e6af5d27a2c0c4efa2fcb31754651481b23eef1fa7ef83f7cb","941351.png":"652086f6077988f10d9ab51d2255b6c82e1689e25f605309c2dffacaad995c8f","942845.png":"4f5665eb8550bdbed43fe64dab1b9c27f2245a95c6d5bf2941a3ac1e0ac9fce3","94361.png":"3eb77f4757a5bd81e26c6a5309e9ae02150329829b26a9a3f873981af6bb61e3","944502.png":"96cad7899d1608b45fbb208904eabdda1e1c0daa34beea8aad59bae3310d7143","945271.png":"3eade4c667aa3375e5d4553c90a29dbe881f8e9fd5e3fbde982a2b03b86e2b02","948729.png":"a4615ab96d9de25ef80b5c7f8340fbd97731a8fec4746ec17fcaa99420ec63f5","962847.png":"8bc2da7266e3983f0b35d744b22b805fac1b41beeac262277b10b7a46cefc1ae","964462.png":"6c6c53a620dae2872a065ad74cdce7663240a1f073a1abda054937439d15c04c","966551.png":"a3f5a7a8cbcd02275bc374cd519815adba0fd6062037ae1bc2db13acb12979be","966699.png":"0cd28e419176675082bc10fa0912373df339a245441787e1dc2645c82223a407","96883.png":"7238c021c54c2b5586702c651825a829904e9c55624d21c8fe8ce06e9d792720","972582.png":"3b7eb28e3800a1d1f24e1987d981216e3ea093d809695d70b2b733122587024c","982198.png":"93ad8f4004f2fa2f61eb942fc2f1ca85c10e2c1ed86098a767147fd7b8da5b2d","98292.png":"5ed6a77fc1de29d2f8507b6a7e7d31d01f0c69d6818e28fb26042bab1ef8d44a","99206.png":"d34bbcc69d52cae8554eb1c1f90df9e3dbd98a91fa80eb0a3719bf559073fe1c","997377.png":"4f71a6524eec84d7fc0004736b068063d770825efcb34ca7d160fc9d30b5fe82"}}''')
assert len(TEST["dhash16"]) == 250 and len(TEST["sha256"]) == 250
TEST_FPS = np.array([np.frombuffer(bytes.fromhex(h), np.uint8) for h in TEST["dhash16"].values()])
TEST_SHA = set(TEST["sha256"].values())
print(len(TEST_FPS), "Bengaluru test frames, and anything within", SAME_CAMERA_BITS, "bits of them, kept out")

In [ ]:
def write_image(img, dst):
    h, w = img.shape[:2]
    if max(h, w) > MAX_SIDE:
        s = MAX_SIDE / max(h, w)
        img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    assert cv2.imwrite(dst, img, [cv2.IMWRITE_JPEG_QUALITY, 95])


def extract_fasdd():
    """The FASDD package from Drive (made on the laptop): {WORK}/fasdd/v3_package/..."""
    root = f"{WORK}/fasdd"
    if not os.path.isdir(f"{root}/v3_package"):
        os.makedirs(root, exist_ok=True)
        subprocess.run(["tar", "-xf", FASDD_PATH, "-C", root], check=True)
    return f"{root}/v3_package", json.load(open(f"{root}/v3_package/fasdd_selection.json"))


In [ ]:
# ---- Picture helpers ----
from IPython.display import Image as _Img, display
COL = {0: (0, 128, 255), 1: (200, 200, 200)}

def sheet(paths, boxes_of, out, cols=4, tile=(400, 300)):
    tiles = []
    for p in paths:
        img = cv2.imread(p)
        for k, x1, y1, x2, y2, *rest in boxes_of(p, img):
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), COL[k], 2)
            txt = NAMES[k] + (f" {rest[0]:.2f}" if rest else "")
            cv2.putText(img, txt, (int(x1), max(14, int(y1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, COL[k], 2)
        tiles.append(cv2.resize(img, tile))
    while len(tiles) % cols:
        tiles.append(np.zeros_like(tiles[0]))
    grid = np.vstack([np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)])
    cv2.imwrite(out, grid, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return out

def label_boxes(p, img):
    H, W = img.shape[:2]
    lab = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    out = []
    for line in open(lab):
        k, cx, cy, w, h = line.split()
        k, cx, cy, w, h = int(k), float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
        out.append((k, cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2))
    return out



In [ ]:
V4_SETTINGS = '# ---- Settings ----\nSIZE = 640\nEPOCHS = 30\nPATIENCE = 8\nBATCH = 32\nWORKERS = 2\nSEED = 26\nBASE_WEIGHTS = "yolo11s.pt"      # Ultralytics\' COCO-pretrained YOLO11-S\nN_NEG_TRAIN = 1500               # Bengaluru frames with no fire added to training\nMAX_SIDE = 1280                  # stored copies are at most this wide\nNEAR_DUP_BITS = 40               # fingerprint distance (of 256) at or under which an image is a near-copy\nVAL_FRACTION = 0.10              # carved from training when the download has no validation split\nFA_TARGET = 0.003                # the alarm threshold keeps false alarms on Bengaluru frames at or under this\nRECALL_BAR = 0.85                # same bar as version 2 (= TARGET_A_DFIRE_RECALL below)\nMIN_IMAGES = 1000                # fewer labelled images than this: not the D-Fire download\nNAMES = ["fire", "smoke"]        # the output classes, in this order\nWORK = "/content/work_fire"\nDRIVE_DIR = "/content/drive/MyDrive/itms_fire_v4"   # version 4: a fresh folder\nV3_DRIVE_DIR = "/content/drive/MyDrive/itms_fire_v3"   # only its Bengaluru frame cache is reused (bengaluru_frames.tar/.json)\nUVH_DRIVE_DIR = "/content/drive/MyDrive/itms_uvh26_finetune"   # the vehicle notebook\'s folder, if it still exists\nUVH_REPO = "iisc-aim/UVH-26"     # otherwise the Bengaluru frames are downloaded from here (Hugging Face)\nN_NEG_VAL = 1000                 # Bengaluru frames with no fire in validation (v3: 300): the threshold is chosen on them\nN_NEG_VAL_BASE = 300             # the first part of them, the same frames as versions 2 and 3\nN_TEST2 = 250                    # FRESH Bengaluru test frames: never in any earlier version; scored once\nN_MINE_POOL = 2500               # extra Bengaluru frames looked at for hard negatives (the rest are thrown away)\nN_HARD = 800                     # the ones with the most flame-coloured area go into training as no-fire images\nSAME_CAMERA_BITS = 72            # a frame this close (of 256 bits) to one of the 250 test frames is that camera\nREQUIRE_BENGALURU = True         # stop if the Bengaluru frames are missing (version 1 carried on without them)\nULTRALYTICS_VERSION = "8.4.163"\nKAGGLE_DATASET = "sayedgamal99/smoke-fire-detection-yolo"       # the "ready to use" copy D-Fire links to\nDFIRE_ONEDRIVE = "https://1drv.ms/u/c/c0bd25b6b048b01d/EbLgD7bES4FDvUN37Grxn8QBF5gIBBc7YV2qklF08GCiBw"\nZIP_NAME = "itms_fire_v4_results.zip"\nFASDD_TAR = "fasdd_v3_package.tar"   # made on the laptop (fasdd_select.py); put it in the Drive folder above\nNEAR_HELDOUT_BITS = 40           # a held-out image this close (of 256) to a training/validation image is dropped from test C\nN_PASTE = 600                    # Bengaluru training frames copied with D-Fire (train) flames pasted in, labelled fire\nMIN_FLAME_CROPS = 100           # fewer usable D-Fire fire boxes than this: something is wrong, stop\nPASTE_MIN_BOX = 32               # smallest D-Fire fire box (pixels, shorter side) used as a flame crop\nPASTE_HEIGHTS = (40, 220)        # pasted flame height range in the (at most 1280 px wide) frame, log-uniform\n# Targets (written down BEFORE the run: itms/baselines/2026-09-30_m8_fire_v4/TARGETS_LOCKED.md)\nTARGET_A_DFIRE_RECALL = 0.85     # D-Fire test images with fire or smoke that alarm\nTARGET_C_STREET_RECALL = 0.80    # FASDD held-out street/vehicle fires (labels V, R) that alarm\nTARGET_B2_MAX_SHARE = 0.01       # fresh Bengaluru test frames that may alarm (250 frames: at most 2)'
V4_GET = '# ---- Get D-Fire (from Drive, Kaggle, or OneDrive) ----\nRAW = f"{WORK}/dfire_raw"\n\ndef _unzip(zp, dest):\n    tmp = dest + ".part"\n    shutil.rmtree(tmp, ignore_errors=True)\n    with zipfile.ZipFile(zp) as z:\n        bad = z.testzip()\n        assert bad is None, f"{zp} is damaged ({bad}); download it again"\n        z.extractall(tmp)\n    os.replace(tmp, dest)\n\ndef acquire():\n    if os.path.isdir(RAW) and any(os.scandir(RAW)):\n        return "already here", RAW\n    zips = [p for p in glob.glob(f"{OUT}/*.zip") if os.path.basename(p) != ZIP_NAME]\n    if zips:\n        _unzip(sorted(zips)[0], RAW)\n        return f"Drive zip {os.path.basename(sorted(zips)[0])}", RAW\n    errors = []\n    try:\n        import kagglehub\n        path = kagglehub.dataset_download(KAGGLE_DATASET)\n        shutil.copytree(path, RAW + ".part", dirs_exist_ok=True)\n        os.replace(RAW + ".part", RAW)\n        return f"Kaggle {KAGGLE_DATASET}", RAW\n    except Exception as e:  # noqa: BLE001\n        errors.append(f"Kaggle: {e}")\n    try:\n        token = base64.urlsafe_b64encode(DFIRE_ONEDRIVE.encode()).decode().rstrip("=")\n        url = f"https://api.onedrive.com/v1.0/shares/u!{token}/root/content"\n        zp = f"{WORK}/dfire_onedrive.zip"\n        with requests.get(url, stream=True, timeout=120, allow_redirects=True) as r:\n            r.raise_for_status()\n            with open(zp + ".part", "wb") as f:\n                for chunk in r.iter_content(1 << 22):\n                    f.write(chunk)\n        os.replace(zp + ".part", zp)\n        _unzip(zp, RAW)\n        return "OneDrive", RAW\n    except Exception as e:  # noqa: BLE001\n        errors.append(f"OneDrive: {e}")\n    raise RuntimeError("Could not get D-Fire automatically:\\n  " + "\\n  ".join(errors) +\n                       f"\\nDownload the zip from https://github.com/gaia-solutions-on-demand/DFireDataset in "\n                       f"your browser, put it in Drive folder {DRIVE_DIR}, then Run all again.")\n\nDS = f"{WORK}/ds"\nMANIFEST = f"{OUT}/fire_manifest.json"\nTAR = f"{OUT}/fire_dataset.tar"\nFASDD_PATH = f"{OUT}/{FASDD_TAR}"\nNEED_BUILD = not (os.path.exists(MANIFEST) and os.path.exists(TAR))\nif not os.path.exists(FASDD_PATH):\n    raise SystemExit(f"STOP: {FASDD_TAR} is not in the Drive folder {DRIVE_DIR}. It is the FASDD street-fire "\n                     "package made on the laptop; version 3 exists to learn street fires from it. Upload it "\n                     "there and Run all again.")\nif NEED_BUILD:\n    SOURCE, RAW = acquire()\n    print("D-Fire from:", SOURCE)\nelse:\n    print("the prepared dataset is in Drive; nothing to download")'
V4_READ = '# ---- Read it: images, labels, splits, and which id is fire ----\ndef scan(root):\n    items, skipped = [], Counter()\n    for dirpath, _, files in os.walk(root):\n        for fn in sorted(files):\n            if not fn.lower().endswith(IMG_EXT) or "__MACOSX" in dirpath:\n                continue\n            p = os.path.join(dirpath, fn)\n            lab = label_path_for(p)\n            if lab is None:\n                skipped["no label file"] += 1\n                continue\n            items.append({"path": p, "label": lab, "split": split_of(os.path.relpath(p, root))})\n    return items, skipped\n\ndef names_file(root):\n    """Class names shipped with the download, if any: {id: name}."""\n    for p in glob.glob(f"{root}/**/*.yaml", recursive=True) + glob.glob(f"{root}/**/*.yml", recursive=True):\n        txt = open(p, errors="replace").read()\n        m = re.search(r"names\\s*:\\s*\\[([^\\]]*)\\]", txt)\n        if m:\n            vals = [v.strip().strip("\'\\"").lower() for v in m.group(1).split(",") if v.strip()]\n            return {i: v for i, v in enumerate(vals)}, os.path.relpath(p, root)\n        m = re.findall(r"^\\s*(\\d+)\\s*:\\s*[\'\\"]?([A-Za-z_ -]+?)[\'\\"]?\\s*$", txt, flags=re.M)\n        if m and "names" in txt:\n            return {int(i): v.strip().lower() for i, v in m}, os.path.relpath(p, root)\n    for p in glob.glob(f"{root}/**/classes.txt", recursive=True):\n        vals = [v.strip().lower() for v in open(p) if v.strip()]\n        return {i: v for i, v in enumerate(vals)}, os.path.relpath(p, root)\n    return None, None\n\nFIRE_WORDS, SMOKE_WORDS = {"fire", "flame", "flames"}, {"smoke"}\n\ndef identify_classes(items):\n    """Which label id is fire: the colour of its boxes, checked against the download\'s names file."""\n    rng = random.Random(SEED)\n    per_class = defaultdict(list)\n    for it in rng.sample(items, min(3000, len(items))):\n        img = cv2.imread(it["path"])\n        if img is None:\n            continue\n        H, W = img.shape[:2]\n        for k, *b in parse_labels(it["label"], W, H)[0]:\n            if len(per_class[k]) < 400:\n                per_class[k].append(flame_fraction(img, b))\n    flame = {k: float(np.median(v)) for k, v in per_class.items() if len(v) >= 20}\n    print("median flame-coloured share of box pixels, by label id:", {k: round(v, 3) for k, v in flame.items()})\n    assert len(flame) == 2, f"expected two label ids with boxes, found {sorted(flame)}"\n    fire_id, smoke_id = max(flame, key=flame.get), min(flame, key=flame.get)\n    assert flame[fire_id] >= 0.05 and flame[fire_id] >= 2 * max(flame[smoke_id], 0.01), (\n        "The box colours do not say which label id is fire. Stop and look at a few images with their labels.")\n    declared, declared_from = names_file(RAW)\n    print("class names file:", declared_from, declared)\n    if declared:\n        assert declared.get(fire_id, "fire") in FIRE_WORDS and declared.get(smoke_id, "smoke") in SMOKE_WORDS, (\n            f"The dataset\'s names file {declared} disagrees with the box colours (fire looks like id {fire_id}).")\n    return {fire_id: NAMES.index("fire"), smoke_id: NAMES.index("smoke")}, flame, declared\n\nif NEED_BUILD:\n    items, scan_skips = scan(RAW)\n    print(len(items), "labelled images;", dict(scan_skips))\n    assert len(items) >= MIN_IMAGES, "too few labelled images: is this the D-Fire download?"\n    print("splits in the download:", dict(Counter(it["split"] for it in items)))\n    ID_MAP, flame, declared = identify_classes(items)\n    print("label ids ->", {k: NAMES[v] for k, v in ID_MAP.items()})'
V4_BUILD = '# ---- Build the training set (or restore it from Drive) ----\nSETTINGS = {"SEED": SEED, "N_NEG_TRAIN": N_NEG_TRAIN, "MAX_SIDE": MAX_SIDE, "NEAR_DUP_BITS": NEAR_DUP_BITS,\n            "VAL_FRACTION": VAL_FRACTION, "N_NEG_VAL": N_NEG_VAL, "SAME_CAMERA_BITS": SAME_CAMERA_BITS,\n            "N_NEG_VAL_BASE": N_NEG_VAL_BASE, "N_TEST2": N_TEST2, "N_MINE_POOL": N_MINE_POOL, "N_HARD": N_HARD,\n            "N_PASTE": N_PASTE, "PASTE_MIN_BOX": PASTE_MIN_BOX, "PASTE_HEIGHTS": list(PASTE_HEIGHTS),\n            "FASDD_TAR": FASDD_TAR}\n\nBLR_INFO = {}\n\ndef _blr_from_hf(dest):\n    """Download Bengaluru frames from UVH-26\'s training images: never one of the 250 test frames (by name or\n    content), never one within SAME_CAMERA_BITS of a test frame (the same camera). Seeded order."""\n    from concurrent.futures import ThreadPoolExecutor\n    files = hf_list()\n    pool = sorted(set(files) - set(TEST["sha256"]))\n    random.Random(SEED).shuffle(pool)\n    need = N_NEG_VAL_BASE + N_NEG_TRAIN\n    for sp in ("train", "val"):\n        os.makedirs(f"{dest}/ds/images/{sp}", exist_ok=True)\n    kept, skipped, i, t0 = [], Counter(), 0, time.time()\n    with ThreadPoolExecutor(8) as ex:\n        while len(kept) < need and i < len(pool):\n            chunk = pool[i:i + 48]\n            i += len(chunk)\n            for name, data in ex.map(lambda n: (n, hf_get(files[n])), chunk):   # order kept: reproducible\n                if len(kept) >= need:\n                    break\n                if data is None:\n                    skipped["file missing"] += 1\n                    continue\n                if hashlib.sha256(data).hexdigest() in TEST_SHA:\n                    skipped["identical to a test frame"] += 1\n                    continue\n                img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)\n                if img is None:\n                    skipped["does not open"] += 1\n                    continue\n                if hamming_min(dhash16(img)[None], TEST_FPS)[0] <= SAME_CAMERA_BITS:\n                    skipped["looks like a test camera"] += 1\n                    continue\n                sp = "val" if len(kept) < N_NEG_VAL_BASE else "train"\n                write_image(img, f"{dest}/ds/images/{sp}/{os.path.splitext(name)[0]}.jpg")\n                kept.append([name, sp])\n            print(f"  Bengaluru frames: {len(kept)}/{need} kept, {sum(skipped.values())} skipped, "\n                  f"{time.time() - t0:.0f} s", flush=True)\n    return {"source": f"Hugging Face {UVH_REPO} UVH-26-Train/data", "pool": len(pool), "kept": kept,\n            "skipped": dict(skipped)}\n\ndef bengaluru_negatives():\n    """Bengaluru traffic frames with no fire: N_NEG_TRAIN for training, N_NEG_VAL for validation (the threshold\n    is chosen on them). From, in order: this notebook\'s own earlier download (Drive), the vehicle notebook\'s\n    dataset.tar (Drive), or IISc\'s UVH-26 on Hugging Face. None of the 250 test frames, nor their cameras: the\n    laptop uses those frames for the final false-alarm count."""\n    dest = f"{WORK}/uvh"\n    cache, cache_info = f"{OUT}/bengaluru_frames.tar", f"{OUT}/bengaluru_frames.json"\n    vehicle_tar = f"{UVH_DRIVE_DIR}/dataset.tar"\n    v3c, v3i = f"{V3_DRIVE_DIR}/bengaluru_frames.tar", f"{V3_DRIVE_DIR}/bengaluru_frames.json"\n    if not (os.path.exists(cache) and os.path.exists(cache_info)) and os.path.exists(v3c) and os.path.exists(v3i):\n        print("reusing version 3\'s Bengaluru frames (the same 1,500 training and 300 validation frames)")\n        shutil.copy(v3c, cache)\n        shutil.copy(v3i, cache_info)\n    if os.path.exists(cache) and os.path.exists(cache_info):\n        BLR_INFO.update(json.load(open(cache_info)))\n        if not os.path.isdir(f"{dest}/ds"):\n            os.makedirs(dest, exist_ok=True)\n            subprocess.run(["tar", "-xf", cache, "-C", dest], check=True)\n    elif os.path.exists(vehicle_tar):\n        os.makedirs(dest, exist_ok=True)\n        subprocess.run(["tar", "-xf", vehicle_tar, "-C", dest], check=True)\n        BLR_INFO.update({"source": f"the vehicle notebook\'s {vehicle_tar} (its training and hold-out frames, "\n                                   "which exclude the test frames\' cameras)"})\n    else:\n        print(f"No Bengaluru frames in Drive ({vehicle_tar}); downloading them from {UVH_REPO} ...")\n        try:\n            info = _blr_from_hf(dest)\n        except Exception as e:  # noqa: BLE001 -- reported, then the REQUIRE check below decides\n            print("the download failed:", repr(e))\n            info = None\n        if info is not None:\n            BLR_INFO.update(info)\n            subprocess.run(["tar", "-cf", cache + ".part", "-C", dest, "ds"], check=True)\n            os.replace(cache + ".part", cache)\n            json.dump(info, open(cache_info, "w"), indent=1)\n    tr = sorted(glob.glob(f"{dest}/ds/images/train/*.jpg"))\n    va = sorted(glob.glob(f"{dest}/ds/images/val/*.jpg"))\n    if len(tr) < N_NEG_TRAIN or not va:\n        if REQUIRE_BENGALURU:\n            raise SystemExit(\n                f"STOP: no Bengaluru frames ({len(tr)} for training, {len(va)} for validation; "\n                f"{N_NEG_TRAIN} and some are needed).\\n"\n                "They are what teaches the model that an orange auto or a hazy road is not a fire, and what the "\n                "alarm threshold is chosen on. Version 1 ran without them and alarmed on 12.8 % of Bengaluru "\n                "traffic frames.\\n"\n                f"Fix: check that this runtime can reach huggingface.co (the download above), or put the vehicle "\n                f"notebook\'s dataset.tar in {UVH_DRIVE_DIR}. Then Run all again.")\n        print("NOTE: no Bengaluru frames - training on D-Fire alone (REQUIRE_BENGALURU is off); the threshold "\n              "is then chosen on D-Fire\'s own negatives.")\n        return [], []\n    random.Random(SEED).shuffle(tr)\n    BLR_INFO.update({"train_used": N_NEG_TRAIN, "val_used": len(va)})\n    return tr[:N_NEG_TRAIN], va\n\nFRESH_INFO = {}\n\ndef orange_score(img):\n    """Largest connected blob of flame-coloured pixels (saturated red-orange-yellow), as a share of the image."""\n    s = 640 / max(img.shape[:2])\n    small = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img\n    h, sat, v = cv2.split(cv2.cvtColor(small, cv2.COLOR_BGR2HSV))\n    m = (((h <= 35) | (h >= 170)) & (sat >= 90) & (v >= 150)).astype(np.uint8)\n    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))\n    n, _, stats, _ = cv2.connectedComponentsWithStats(m)\n    return float(stats[1:, cv2.CC_STAT_AREA].max()) / m.size if n > 1 else 0.0\n\ndef _fresh_from_hf(dest, used_stems, base_fps, n_val_have):\n    """More Bengaluru frames from UVH-26, in the same seeded order as `_blr_from_hf`, skipping the frames already used\n    (by name, and any within 8 bits of one), the 250 old test frames (by name, content and camera). Roles in order:\n    extra validation, FRESH test frames, the pool the hard negatives are picked from."""\n    from concurrent.futures import ThreadPoolExecutor\n    files = hf_list()\n    pool = sorted(set(files) - set(TEST["sha256"]))\n    random.Random(SEED).shuffle(pool)\n    pool = [n for n in pool if os.path.splitext(n)[0] not in used_stems]\n    roles = [("val", max(0, N_NEG_VAL - n_val_have)), ("test2", N_TEST2), ("pool", N_MINE_POOL)]\n    for r, _ in roles:\n        os.makedirs(f"{dest}/{r}", exist_ok=True)\n    need, kept, skipped, i, t0 = sum(n for _, n in roles), [], Counter(), 0, time.time()\n    bank = base_fps\n    with ThreadPoolExecutor(8) as ex:\n        while len(kept) < need and i < len(pool):\n            chunk = pool[i:i + 48]\n            i += len(chunk)\n            for name, data in ex.map(lambda n: (n, hf_get(files[n])), chunk):\n                if len(kept) >= need:\n                    break\n                if data is None:\n                    skipped["file missing"] += 1\n                    continue\n                if hashlib.sha256(data).hexdigest() in TEST_SHA:\n                    skipped["identical to an old test frame"] += 1\n                    continue\n                img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)\n                if img is None:\n                    skipped["does not open"] += 1\n                    continue\n                fp = dhash16(img)\n                if hamming_min(fp[None], TEST_FPS)[0] <= SAME_CAMERA_BITS:\n                    skipped["looks like an old test camera"] += 1\n                    continue\n                if len(bank) and hamming_min(fp[None], bank)[0] <= 8:\n                    skipped["duplicate of a frame already used"] += 1\n                    continue\n                bank = np.vstack([bank, fp[None]]) if len(bank) else fp[None]\n                k, acc = len(kept), 0\n                for r, n in roles:\n                    if k < acc + n:\n                        role = r\n                        break\n                    acc += n\n                write_image(img, f"{dest}/{role}/{os.path.splitext(name)[0]}.jpg")\n                kept.append([name, role])\n            print(f"  fresh Bengaluru frames: {len(kept)}/{need} kept, {sum(skipped.values())} skipped, "\n                  f"{time.time() - t0:.0f} s", flush=True)\n    return {"source": f"Hugging Face {UVH_REPO} UVH-26-Train/data", "kept": kept, "skipped": dict(skipped)}\n\ndef _fps(paths):\n    return np.stack([dhash16(cv2.imread(p)) for p in paths]) if paths else np.zeros((0, 32), np.uint8)\n\ndef fresh_bengaluru(neg_tr, neg_va):\n    """The extra validation frames, the FRESH test frames and the hard negatives. Any training, validation or hard-negative\n    frame within SAME_CAMERA_BITS of a fresh test frame (the same camera) is dropped from training, so the test cameras\n    are unseen. Needs the network the first time (then it is cached in Drive)."""\n    if not neg_tr and not neg_va:   # REQUIRE_BENGALURU is off and there is no base: nothing to extend\n        FRESH_INFO.update({"skipped": "no Bengaluru base frames"})\n        return [], [], [], []\n    dest = f"{WORK}/uvh_fresh"\n    cache, cache_info = f"{OUT}/bengaluru_fresh.tar", f"{OUT}/bengaluru_fresh.json"\n    used = {os.path.splitext(os.path.basename(p))[0] for p in list(neg_tr) + list(neg_va)}\n    if os.path.exists(cache) and os.path.exists(cache_info):\n        info = json.load(open(cache_info))\n        if not os.path.isdir(dest):\n            os.makedirs(dest, exist_ok=True)\n            subprocess.run(["tar", "-xf", cache, "-C", dest], check=True)\n    else:\n        try:\n            info = _fresh_from_hf(dest, used, _fps(list(neg_tr) + list(neg_va)), len(neg_va))\n        except Exception as e:  # noqa: BLE001\n            raise SystemExit(f"STOP: could not get the fresh Bengaluru test frames ({e!r}). Version 4 needs them (they "\n                             f"are its test); check that this runtime can reach huggingface.co and Run all again.")\n        subprocess.run(["tar", "-cf", cache + ".part", "-C", dest, "val", "test2", "pool"], check=True)\n        os.replace(cache + ".part", cache)\n        json.dump(info, open(cache_info, "w"), indent=1)\n    extra_val = sorted(glob.glob(f"{dest}/val/*.jpg"))\n    test2 = sorted(glob.glob(f"{dest}/test2/*.jpg"))\n    pool = sorted(glob.glob(f"{dest}/pool/*.jpg"))\n    assert len(test2) == N_TEST2, f"{len(test2)} fresh test frames, expected {N_TEST2}"\n    assert len(extra_val) == max(0, N_NEG_VAL - len(neg_va)), (len(extra_val), N_NEG_VAL, len(neg_va))\n    scores_ = sorted(((orange_score(cv2.imread(p)), p) for p in pool), reverse=True)\n    hard = [p for _, p in scores_[:N_HARD]]\n    t2fp = _fps(test2)\n    dropped = Counter()\n\n    def clean(paths, name):\n        if not paths or not len(t2fp):\n            return list(paths)\n        near = hamming_min(_fps(paths), t2fp)\n        # hamming_min gives the smallest distance from each path to any test frame\n        keep = [p for p, d in zip(paths, near) if d > SAME_CAMERA_BITS]\n        dropped[name] = len(paths) - len(keep)\n        return keep\n    neg_tr, neg_va = clean(list(neg_tr), "train"), clean(list(neg_va), "val")\n    extra_val, hard = clean(extra_val, "extra val"), clean(hard, "hard negatives")\n    FRESH_INFO.update({"source": info["source"], "skipped": info["skipped"], "test2": [os.path.basename(p) for p in test2],\n                       "dropped_as_same_camera_as_a_fresh_test_frame": dict(dropped),\n                       "hard_pool": len(pool), "hard_kept": len(hard),\n                       "hard_score_range": [round(scores_[N_HARD - 1][0], 4), round(scores_[0][0], 4)] if len(scores_) >= N_HARD else None,\n                       "extra_val": len(extra_val)})\n    return extra_val, hard, neg_tr, neg_va\n\ndef write_image(img, dst):\n    h, w = img.shape[:2]\n    if max(h, w) > MAX_SIDE:\n        s = MAX_SIDE / max(h, w)\n        img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)\n    assert cv2.imwrite(dst, img, [cv2.IMWRITE_JPEG_QUALITY, 95])\n\ndef extract_fasdd():\n    """The FASDD package from Drive (made on the laptop): {WORK}/fasdd/v3_package/..."""\n    root = f"{WORK}/fasdd"\n    if not os.path.isdir(f"{root}/v3_package"):\n        os.makedirs(root, exist_ok=True)\n        subprocess.run(["tar", "-xf", FASDD_PATH, "-C", root], check=True)\n    return f"{root}/v3_package", json.load(open(f"{root}/v3_package/fasdd_selection.json"))\n\nFASDD_IDS = {0: 0, 1: 1}     # FASDD\'s YOLO labels: 0 = fire, 1 = smoke (checked by colour below, not trusted)\n\ndef import_fasdd(bank, test_sha, skipped, counts, names_out):\n    """Add the FASDD images to the dataset. Dropped: any that is identical to, or within NEAR_DUP_BITS of, a\n    D-Fire test image (FASDD reuses older datasets; D-Fire\'s test images must stay unseen). The label ids are\n    checked by colour, like D-Fire\'s."""\n    pkg, sel = extract_fasdd()\n    rows, colour = [], defaultdict(list)\n    for m in sel["images"]:\n        p = f"{pkg}/images/{m[\'split\']}/{m[\'stem\']}.jpg"\n        img = cv2.imread(p)\n        if img is None:\n            skipped["FASDD: does not open"].append(p)\n            continue\n        H, W = img.shape[:2]\n        boxes, st = parse_labels(f"{pkg}/labels/{m[\'split\']}/{m[\'stem\']}.txt", W, H)\n        rows.append((m, p, None, boxes, dhash16(img), hashlib.sha256(open(p, "rb").read()).hexdigest()))\n        if len(colour[0]) < 400 or len(colour[1]) < 400:\n            for k, *b in boxes:\n                if k in (0, 1) and len(colour[k]) < 400:\n                    colour[k].append(flame_fraction(img, b))\n    fire_c, smoke_c = (float(np.median(colour[k])) if colour[k] else 0.0 for k in (0, 1))\n    print("FASDD median flame-coloured share of box pixels, id 0 / id 1:", round(fire_c, 3), round(smoke_c, 3))\n    assert fire_c >= 0.05 and fire_c >= 2 * max(smoke_c, 0.01), (\n        "FASDD label id 0 does not look like fire by colour: stop and look at a few images with their labels.")\n    dist = hamming_min(np.stack([r[4] for r in rows]), bank)\n    kept_n = Counter()\n    for (m, p, img, boxes, fp, sha), d in zip(rows, dist):\n        sp = m["split"]\n        if sha in test_sha:\n            skipped["FASDD: identical to a D-Fire test image"].append(p)\n            continue\n        if d <= NEAR_DUP_BITS:\n            skipped["FASDD: near-copy of a D-Fire test image"].append(p)\n            continue\n        stem = m["stem"]\n        img = cv2.imread(p)\n        write_image(img, f"{DS}/images/{sp}/{stem}.jpg")\n        H, W = img.shape[:2]\n        with open(f"{DS}/labels/{sp}/{stem}.txt", "w") as f:\n            for k, x1, y1, x2, y2 in boxes:\n                if k not in FASDD_IDS:\n                    continue\n                c = FASDD_IDS[k]\n                f.write(f"{c} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} "\n                        f"{(y2 - y1) / H:.6f}\\n")\n                counts[sp][NAMES[c]] += 1\n        counts[sp]["images"] += 1\n        counts[sp]["FASDD images"] += 1\n        counts[sp]["images with fire or smoke"] += 1\n        kept_n[f"{sp}_{m[\'kind\']}"] += 1\n        names_out.append({"stem": stem, "split": sp, "source": "fasdd:" + m["file"], "sha256": sha})\n    return {"selection_settings": sel["settings"], "selection_stats": sel["stats"], "kept": dict(kept_n),\n            "median_flame_share_id0_id1": [fire_c, smoke_c]}\n\ndef feathered(patch, bg, x, y, edge=0.15):\n    """`patch` pasted into a copy of `bg` at (x, y), fading out over `edge` of its size."""\n    h, w = patch.shape[:2]\n    ry = np.minimum(np.arange(h), np.arange(h)[::-1]) / max(1.0, edge * h)\n    rx = np.minimum(np.arange(w), np.arange(w)[::-1]) / max(1.0, edge * w)\n    a = np.clip(np.minimum.outer(ry, rx), 0, 1)[..., None]\n    out = bg.copy()\n    roi = out[y:y + h, x:x + w].astype(np.float32)\n    out[y:y + h, x:x + w] = (a * patch + (1 - a) * roi).astype(np.uint8)\n    return out\n\ndef paste_flames(dfire_fire, neg_tr, counts, names_out):\n    """N_PASTE copies of Bengaluru training frames with 1-2 flames pasted in (crops of D-Fire TRAIN fire boxes),\n    labelled as fire. The plain frame stays in training too, so the same street is seen with and without a fire.\n    Validation frames are never pasted into: the alarm threshold is chosen on plain ones."""\n    rng = random.Random(SEED + 7)\n    pool = [c for c in dfire_fire if min(c[1][2] - c[1][0], c[1][3] - c[1][1]) >= PASTE_MIN_BOX]\n    assert len(pool) >= MIN_FLAME_CROPS, f"only {len(pool)} usable D-Fire fire boxes for flame crops"\n    frames = list(neg_tr)\n    rng.shuffle(frames)\n    made = 0\n    for bg_path in frames[:N_PASTE]:\n        bg = cv2.imread(bg_path)\n        H, W = bg.shape[:2]\n        img, labs = bg, []\n        for _ in range(rng.choice((1, 1, 2))):\n            stem, (x1, y1, x2, y2), (ow, oh) = pool[rng.randrange(len(pool))]\n            src = cv2.imread(f"{DS}/images/train/{stem}.jpg")\n            sh, sw = src.shape[:2]\n            fx, fy = sw / ow, sh / oh\n            bw, bh = (x2 - x1) * fx, (y2 - y1) * fy\n            mx, my = 0.1 * bw, 0.1 * bh\n            cx1, cy1 = max(0, int(x1 * fx - mx)), max(0, int(y1 * fy - my))\n            cx2, cy2 = min(sw, int(x2 * fx + mx)), min(sh, int(y2 * fy + my))\n            patch0 = src[cy1:cy2, cx1:cx2]\n            if patch0.size == 0:\n                continue\n            if rng.random() < 0.5:\n                patch0 = patch0[:, ::-1]\n            h = int(round(np.exp(rng.uniform(np.log(PASTE_HEIGHTS[0]), np.log(PASTE_HEIGHTS[1])))))\n            w = min(3 * h, max(8, round(patch0.shape[1] * h / patch0.shape[0])))\n            if w >= 0.9 * W or h >= 0.6 * H:\n                continue\n            patch = cv2.resize(np.ascontiguousarray(patch0), (w, h), interpolation=cv2.INTER_AREA)\n            x = rng.randrange(int(0.05 * W), max(int(0.05 * W) + 1, int(0.95 * W) - w))\n            y = rng.randrange(int(0.35 * H), max(int(0.35 * H) + 1, int(0.95 * H) - h))\n            img = feathered(patch, img, x, y)\n            # label the flame itself (the patch without the margin), as a fraction of the frame\n            ix1, iy1 = x + w * (x1 * fx - cx1) / (cx2 - cx1), y + h * (y1 * fy - cy1) / (cy2 - cy1)\n            ix2, iy2 = x + w * (x2 * fx - cx1) / (cx2 - cx1), y + h * (y2 * fy - cy1) / (cy2 - cy1)\n            labs.append(((ix1 + ix2) / 2 / W, (iy1 + iy2) / 2 / H, (ix2 - ix1) / W, (iy2 - iy1) / H))\n        if not labs:\n            continue\n        stem = "blrp_" + os.path.splitext(os.path.basename(bg_path))[0]\n        write_image(img, f"{DS}/images/train/{stem}.jpg")\n        with open(f"{DS}/labels/train/{stem}.txt", "w") as f:\n            for cx, cy, bw_, bh_ in labs:\n                f.write(f"0 {cx:.6f} {cy:.6f} {bw_:.6f} {bh_:.6f}\\n")\n        counts["train"]["images"] += 1\n        counts["train"]["Bengaluru frames with pasted flames"] += 1\n        counts["train"]["images with fire or smoke"] += 1\n        counts["train"]["fire"] += len(labs)\n        names_out.append({"stem": stem, "split": "train", "source": "pasted:" + os.path.basename(bg_path)})\n        made += 1\n    return {"frames": made, "flame_crops_available": len(pool)}\n\ndef build():\n    for d in ("images/train", "images/val", "images/test", "labels/train", "labels/val", "labels/test"):\n        os.makedirs(f"{DS}/{d}", exist_ok=True)\n    skipped, box_stats = defaultdict(list), Counter()\n    for k, n in scan_skips.items():\n        skipped[k].extend([None] * n)       # counted during the scan (no paths kept)\n    recs = []   # (item, fingerprint, checksum, boxes)\n    for it in items:\n        img = cv2.imread(it["path"])\n        if img is None:\n            skipped["does not open"].append(it["path"])\n            continue\n        H, W = img.shape[:2]\n        boxes, st = parse_labels(it["label"], W, H)\n        box_stats.update(st)\n        unknown = [b for b in boxes if b[0] not in ID_MAP]\n        if unknown:\n            box_stats["box with an unknown class id dropped"] += len(unknown)\n            boxes = [b for b in boxes if b[0] in ID_MAP]\n        recs.append((it, dhash16(img), hashlib.sha256(open(it["path"], "rb").read()).hexdigest(), boxes))\n    # splits: the download\'s own; validation carved from training when it has none (seeded)\n    have = {r[0]["split"] for r in recs}\n    split_note = "the download\'s train / val / test split"\n    if "test" not in have:\n        # no split in the download: a seeded one that keeps each group of near-copies on one side\n        group = near_copy_groups(np.stack([r[1] for r in recs]), NEAR_DUP_BITS)\n        members = defaultdict(list)\n        for i, gid in enumerate(group):\n            members[gid].append(i)\n        gids = sorted(members)\n        biggest = max(len(v) for v in members.values())\n        if biggest > 0.1 * len(recs):\n            print(f"WARNING: one group of near-copies holds {biggest} of {len(recs)} images; the split is lumpy")\n        random.Random(SEED).shuffle(gids)\n        targets = [("test", 0.2 * len(recs)), ("val", VAL_FRACTION * len(recs))]\n        phase, filled = 0, Counter()\n        for gid in gids:           # test fills first, then validation (each gets at least one group), then train\n            if phase < 2 and filled[targets[phase][0]] >= targets[phase][1]:\n                phase += 1\n            sp = targets[phase][0] if phase < 2 else "train"\n            for i in members[gid]:\n                recs[i][0]["split"] = sp\n            filled[sp] += len(members[gid])\n        have = {r[0]["split"] for r in recs}\n        split_note = (f"no split in the download: a seeded one by groups of near-copies ({len(gids)} groups), "\n                      f"test 20 %, validation {VAL_FRACTION:.0%}")\n    if "val" not in have:\n        tr = [i for i, r in enumerate(recs) if r[0]["split"] == "train"]\n        random.Random(SEED).shuffle(tr)\n        for i in tr[:int(len(tr) * VAL_FRACTION)]:\n            recs[i][0]["split"] = "val"\n        split_note = f"the download\'s train / test split; validation = a seeded {VAL_FRACTION:.0%} of train"\n    orphans = [r for r in recs if r[0]["split"] is None]\n    for r in orphans:\n        skipped["not in a train / val / test folder"].append(r[0]["path"])\n    recs = [r for r in recs if r[0]["split"] is not None]\n    # keep the test images, and near-copies of them, out of training and validation\n    test = [r for r in recs if r[0]["split"] == "test"]\n    test_sha = {r[2] for r in test}\n    bank = np.stack([r[1] for r in test])\n    others = [r for r in recs if r[0]["split"] != "test"]\n    dist = hamming_min(np.stack([r[1] for r in others]), bank)\n    kept = list(test)\n    for r, d in zip(others, dist):\n        if r[2] in test_sha:\n            skipped[f"{r[0][\'split\']}: identical to a test image"].append(r[0]["path"])\n        elif d <= NEAR_DUP_BITS:\n            skipped[f"{r[0][\'split\']}: near-copy of a test image"].append(r[0]["path"])\n        else:\n            kept.append(r)\n    # training images that are near-copies of validation images are counted, not dropped: validation only\n    # picks the best epoch, and the alarm threshold is set by false alarms on Bengaluru frames, not by recall\n    val = [r for r in kept if r[0]["split"] == "val"]\n    train = [r for r in kept if r[0]["split"] == "train"]\n    near_val = 0\n    if val and train:\n        near_val = int((hamming_min(np.stack([r[1] for r in train]), np.stack([r[1] for r in val]))\n                        <= NEAR_DUP_BITS).sum())\n    n_before = sum(1 for r in recs if r[0]["split"] == "train")\n    n_after = sum(1 for r in kept if r[0]["split"] == "train")\n    assert n_after > 0, "every training image is a near-copy of a test image: the split is unusable"\n    if n_after < 0.75 * n_before:\n        print(f"WARNING: keeping the test clean removed {n_before - n_after} of {n_before} training images "\n              "(frames of the same videos on both sides of the download\'s split)")\n    names_out, counts = [], {sp: Counter() for sp in ("train", "val", "test")}\n    dfire_train_fire = []   # (stem, box in original pixels, (W, H)): flame crops come from D-Fire TRAIN only\n    for n, (it, _, sha, boxes) in enumerate(kept):\n        sp = it["split"]\n        stem = f"df{n:05d}"\n        img = cv2.imread(it["path"])\n        H, W = img.shape[:2]\n        write_image(img, f"{DS}/images/{sp}/{stem}.jpg")\n        with open(f"{DS}/labels/{sp}/{stem}.txt", "w") as f:\n            for k, x1, y1, x2, y2 in boxes:\n                c = ID_MAP[k]\n                f.write(f"{c} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} "\n                        f"{(y2 - y1) / H:.6f}\\n")\n                counts[sp][NAMES[c]] += 1\n        counts[sp]["images"] += 1\n        counts[sp]["images with fire or smoke" if boxes else "images with neither"] += 1\n        if sp == "train":\n            dfire_train_fire += [(stem, (x1, y1, x2, y2), (W, H)) for k, x1, y1, x2, y2 in boxes if ID_MAP[k] == 0]\n        names_out.append({"stem": stem, "split": sp, "source": os.path.relpath(it["path"], RAW), "sha256": sha})\n    # ---- FASDD street and vehicle fires (chosen on the laptop), minus near-copies of D-Fire\'s test images ----\n    fasdd = import_fasdd(bank, test_sha, skipped, counts, names_out)\n    neg_tr, neg_va = bengaluru_negatives()\n    extra_val, hard, neg_tr, neg_va = fresh_bengaluru(neg_tr, neg_va)\n    neg_va = list(neg_va) + list(extra_val)\n    for sp, files in (("train", neg_tr), ("val", neg_va)):\n        for p in files:\n            stem = "blr_" + os.path.splitext(os.path.basename(p))[0]\n            shutil.copy(p, f"{DS}/images/{sp}/{stem}.jpg")\n            open(f"{DS}/labels/{sp}/{stem}.txt", "w").close()\n            counts[sp]["Bengaluru frames with no fire"] += 1\n            names_out.append({"stem": stem, "split": sp, "source": "uvh26:" + os.path.basename(p)})\n    for p in hard:\n        stem = "blrh_" + os.path.splitext(os.path.basename(p))[0]\n        shutil.copy(p, f"{DS}/images/train/{stem}.jpg")\n        open(f"{DS}/labels/train/{stem}.txt", "w").close()\n        counts["train"]["Bengaluru hard negatives (flame-coloured)"] += 1\n        names_out.append({"stem": stem, "split": "train", "source": "uvh26-hard:" + os.path.basename(p)})\n    pasted = paste_flames(dfire_train_fire, neg_tr, counts, names_out)\n    assert counts["val"]["images with fire or smoke"] > 0, "no validation image has fire or smoke"\n    man = {"settings": dict(SETTINGS), "source": SOURCE, "split": split_note, "id_map": {str(k): NAMES[v]\n           for k, v in ID_MAP.items()}, "flame_share_by_id": flame, "declared_names": declared,\n           "counts": {sp: dict(c) for sp, c in counts.items()}, "box_stats": dict(box_stats),\n           "skipped": {k: len(v) for k, v in skipped.items()},\n           "train_near_copies_of_validation_kept": near_val,\n           "skipped_examples": {k: [p for p in v if p][:20] for k, v in skipped.items()},\n           "bengaluru": {k: v for k, v in BLR_INFO.items() if k != "kept"}, "fasdd": fasdd, "pasted": pasted, "fresh": dict(FRESH_INFO),\n           "images": names_out}\n    json.dump(man, open(MANIFEST, "w"), indent=1)\n    subprocess.run(["tar", "-cf", TAR + ".part", "-C", WORK, "ds"], check=True)\n    os.replace(TAR + ".part", TAR)\n    return man\n\nif os.path.exists(MANIFEST) and os.path.exists(TAR):\n    man = json.load(open(MANIFEST))\n    assert man["settings"] == SETTINGS, (\n        f"The dataset in Drive was made with {man[\'settings\']}, these settings are {SETTINGS}. Put them back, "\n        "or move fire_dataset.tar and fire_manifest.json out of the Drive folder to build a new one.")\n    if not os.path.isdir(DS):\n        print("restoring the prepared dataset from Drive...")\n        subprocess.run(["tar", "-xf", TAR, "-C", WORK], check=True)\nelse:\n    man = build()\nn_blr = {sp: man["counts"].get(sp, {}).get("Bengaluru frames with no fire", 0) for sp in ("train", "val")}\nif REQUIRE_BENGALURU and not (n_blr["train"] and n_blr["val"]):\n    raise SystemExit(f"STOP: the dataset in {OUT} has no Bengaluru frames with no fire ({n_blr}); version 1 "\n                     "carried on like this and alarmed on 12.8 % of Bengaluru traffic frames. Move "\n                     "fire_dataset.tar and fire_manifest.json out of that folder, make sure dataset.tar is in "\n                     f"{UVH_DRIVE_DIR}, and Run all again.")\nfor sp in ("train", "val", "test"):\n    on_disk = len(os.listdir(f"{DS}/images/{sp}"))\n    assert on_disk == sum(1 for r in man["images"] if r["split"] == sp) == len(os.listdir(f"{DS}/labels/{sp}"))\nprint("split:", man["split"])\nprint("counts:", json.dumps(man["counts"], indent=1))\nprint("skipped:", man["skipped"])\nprint("boxes:", man["box_stats"])'
# ---- Find the version 4 files; rebuild the dataset from scratch if it is not in Drive ----
V4 = V4_DRIVE_DIR
PT = f"{V4_DRIVE_DIR}/{PT_NAME}"
if not os.path.exists(PT):
    raise SystemExit(f"STOP: {PT_NAME} (the version 4 model) is not in the Drive folder {V4_DRIVE_DIR}.")
REF_MANIFEST = f"{V4_DRIVE_DIR}/fire_manifest.json"

def find_fasdd_package():
    """The FASDD package: the .tar, or the unpacked v3_package folder (any folder named v3_package*), as a .tar."""
    tar = f"{V4_DRIVE_DIR}/{FASDD_TAR}"
    if os.path.exists(tar):
        return tar
    import tarfile
    for d in sorted(glob.glob(f"{V4_DRIVE_DIR}/v3_package*")):
        if os.path.isdir(d) and os.path.exists(f"{d}/fasdd_selection.json"):
            sel = json.load(open(f"{d}/fasdd_selection.json"))
            count = lambda p: sum(len(fs) for _, _, fs in os.walk(p))
            n_img, n_lab, n_held = count(f"{d}/images"), count(f"{d}/labels"), count(f"{d}/heldout")
            n_sel, n_hsel = len(sel["images"]), len(sel["heldout"])
            assert n_img == n_sel == n_lab, f"{d}: {n_img} images, {n_lab} labels, expected {n_sel}: the upload is incomplete"
            assert n_held >= n_hsel, f"{d}: heldout has {n_held}, expected {n_hsel}: the upload is incomplete"
            out = f"{WORK}/{FASDD_TAR}"
            if not os.path.exists(out):
                print("packing the FASDD folder into a tar...")
                with tarfile.open(out + ".part", "w") as t:
                    t.add(d, arcname="v3_package")
                os.replace(out + ".part", out)
            return out
    return None

FASDD_PATH = find_fasdd_package()
have = {f: os.path.exists(f"{V4_DRIVE_DIR}/{f}") for f in ("fire_dataset.tar", "fire_manifest.json", "bengaluru_fresh.tar")}
if all(have.values()):
    print("the version 4 dataset is in Drive; nothing to rebuild")
else:
    print("not in the Drive folder:", [f for f, ok in have.items() if not ok], "-> rebuilding the version 4 dataset with version 4's own code")
    if FASDD_PATH is None:
        raise SystemExit(f"STOP: {FASDD_TAR} (or the unpacked v3_package folder) is not in {V4_DRIVE_DIR}.")
    if not os.path.exists(REF_MANIFEST):
        raise SystemExit(f"STOP: fire_manifest.json (from capstone/itms_fire_v4_results/) is not in {V4_DRIVE_DIR}. "
                         "The rebuilt dataset is checked against it.")
    REBUILD_DIR = f"{OUT}/v4_rebuild"
    os.makedirs(REBUILD_DIR, exist_ok=True)
    if not os.path.exists(f"{REBUILD_DIR}/{FASDD_TAR}"):
        shutil.copy(FASDD_PATH, f"{REBUILD_DIR}/{FASDD_TAR}")
    ns = dict(globals())
    exec(V4_SETTINGS, ns)       # version 4's own settings ...
    ns.update({"OUT": REBUILD_DIR, "DRIVE_DIR": REBUILD_DIR, "WORK": WORK, "HF_TOKEN": HF_TOKEN, **REBUILD_OVERRIDES})
    for title, text in (("get D-Fire", V4_GET), ("read D-Fire", V4_READ), ("build", V4_BUILD)):
        print("== version 4 step:", title)
        exec(text, ns)
    new, ref = ns["man"], json.load(open(REF_MANIFEST))
    rows = lambda m: {(r["stem"], r["split"]): r.get("sha256") for r in m["images"]}
    a, b = rows(ref), rows(new)
    only_ref, only_new = sorted(set(a) - set(b)), sorted(set(b) - set(a))
    sha_diff = [k for k in set(a) & set(b) if a[k] != b[k]]
    t2 = ref["fresh"]["test2"] == new["fresh"]["test2"]
    if only_ref or only_new or sha_diff or not t2:
        raise SystemExit(f"STOP: the rebuilt dataset is not the version 4 dataset (in the old one only: {len(only_ref)} {only_ref[:5]}; "
                         f"in the new one only: {len(only_new)} {only_new[:5]}; checksums differ: {len(sha_diff)}; "
                         f"same fresh test frames: {t2}). A validation frame could have been a version 4 training frame, "
                         "so there is no honest test. Tell Claude.")
    print(f"the rebuilt dataset has the same {len(a)} images, splits and checksums, and the same {len(ref['fresh']['test2'])} fresh test frames, as version 4")
    V4 = REBUILD_DIR


In [ ]:
# ---- Restore the version 4 dataset, weights and FASDD package from Drive ----
from ultralytics import YOLO
need = ["fire_dataset.tar", "fire_manifest.json", "bengaluru_fresh.tar"]
missing = [f for f in need if not os.path.exists(f"{V4}/{f}")]
if missing:
    raise SystemExit(f"STOP: not in {V4}: {missing}.")
assert sha256_file(PT) == EXPECTED_PT_SHA256, "this is not the version 4 model"
DS = f"{WORK}/ds"
man = json.load(open(f"{V4}/fire_manifest.json"))
if not os.path.isdir(DS):
    print("restoring the version 4 dataset...")
    subprocess.run(["tar", "-xf", f"{V4}/fire_dataset.tar", "-C", WORK], check=True)
for sp in ("train", "val", "test"):
    on_disk = len(os.listdir(f"{DS}/images/{sp}"))
    assert on_disk == sum(1 for r in man["images"] if r["split"] == sp) == len(os.listdir(f"{DS}/labels/{sp}")), sp
REPORT_PATH = f"{OUT}/fire_report.json"
report = json.load(open(REPORT_PATH)) if os.path.exists(REPORT_PATH) else {}
report.update({"ultralytics": ULTRALYTICS_VERSION, "torch": torch.__version__, "v4_pt_sha256": EXPECTED_PT_SHA256,
               "settings": {"FA_TARGET": FA_TARGET, "MIN_RECALL_IMAGES": MIN_RECALL_IMAGES, "N_TEST3": N_TEST3,
                            "SAME_CAMERA_BITS": SAME_CAMERA_BITS, "NEAR_DUP_BITS": NEAR_DUP_BITS,
                            "MAX_CANDIDATES": MAX_CANDIDATES}})

def save_report():
    json.dump(report, open(REPORT_PATH + ".part", "w"), indent=1)
    os.replace(REPORT_PATH + ".part", REPORT_PATH)

best = YOLO(PT)
assert [best.names[i] for i in sorted(best.names)] == NAMES, best.names
bn = bn_report(best.model)
report["bn"] = bn
assert bn["nonfinite_channels"] == 0 and bn["channels_at_fp16_max"] == 0, bn
net = YOLO(PT).model.float().eval()
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net.to(DEV)

def scores(paths):
    """Each image's highest fire and smoke score, from the raw head (no NMS needed for 'is there any')."""
    out = np.zeros((len(paths), len(NAMES)), np.float32)
    with torch.no_grad():
        for i, p in enumerate(paths):
            x, _, _ = to_input(cv2.imread(p), SIZE)
            y = net(torch.from_numpy(x).to(DEV))
            y = (y[0] if isinstance(y, (list, tuple)) else y)[0]
            out[i] = y[4:4 + len(NAMES)].amax(1).float().cpu().numpy()
    return out

def split_table(sp):
    rows = [r for r in man["images"] if r["split"] == sp]
    paths = [f"{DS}/images/{sp}/{r['stem']}.jpg" for r in rows]
    has = np.zeros((len(rows), len(NAMES)), bool)
    for i, r in enumerate(rows):
        for line in open(f"{DS}/labels/{sp}/{r['stem']}.txt"):
            has[i, int(line.split()[0])] = True
    blr = np.array([r["stem"].startswith("blr_") for r in rows])
    return paths, has, blr



In [ ]:
# ---- Fingerprints of every training and validation image (cached in Drive) ----
FP_CACHE = f"{OUT}/fp_trainval.npz"
tv_rows = [(r["split"], r["stem"]) for r in man["images"] if r["split"] in ("train", "val")]
if os.path.exists(FP_CACHE):
    z = np.load(FP_CACHE, allow_pickle=False)
    ok = (list(z["split"]) == [a for a, _ in tv_rows]) and (list(z["stem"]) == [b for _, b in tv_rows])
else:
    ok = False
if not ok:
    t0 = time.time()
    fp = np.stack([dhash16(cv2.imread(f"{DS}/images/{sp}/{st}.jpg")) for sp, st in tv_rows])
    np.savez(FP_CACHE + ".part.npz", split=np.array([a for a, _ in tv_rows]), stem=np.array([b for _, b in tv_rows]), fp=fp)
    os.replace(FP_CACHE + ".part.npz", FP_CACHE)
    z = np.load(FP_CACHE, allow_pickle=False)
    print(f"fingerprinted {len(tv_rows)} images in {time.time() - t0:.0f} s")
TV_SPLIT, TV_STEM, TV_FP = z["split"], z["stem"], z["fp"]
FP_BY = {(a, b): i for i, (a, b) in enumerate(zip(TV_SPLIT, TV_STEM))}
print("fingerprints:", TV_FP.shape)


In [ ]:
# ---- How far is validation from training? Recall of the v5 thresholds on "unseen-scene" validation images ----
# Diagnostic only. Nothing is chosen or trained. D-Fire test images are only fingerprinted (distances to training), never scored here.
assert "choice" in report, "fire_report.json with the v5 Step 1 choice is not in the Drive folder " + OUT
TF, TS = report["choice"]["chosen"]["t_fire"], report["choice"]["chosen"]["t_smoke"]
T1 = report["choice"]["single_threshold_best"]["t_fire"]
BUCKETS = [("0-40 bits (near-copy of a training image)", 0, 40), ("41-72 bits (probably the same video)", 41, 72), (">72 bits (an unseen scene)", 73, 256)]
vp_, vhas_, vblr_ = split_table("val")
vstem_ = [os.path.basename(p)[:-4] for p in vp_]
vs_ = scores(vp_)
tr_idx_ = [i for i in range(len(TV_STEM)) if TV_SPLIT[i] == "train" and not str(TV_STEM[i]).startswith(("blr_", "blrh_", "blrp_"))]
trfp_ = TV_FP[tr_idx_]
vfp_ = np.stack([TV_FP[FP_BY[("val", s)]] for s in vstem_])
dval_ = hamming_min(vfp_, trfp_)
al_ = (vs_[:, 0] >= TF) | (vs_[:, 1] >= TS)
al1_ = (vs_[:, 0] >= T1) | (vs_[:, 1] >= T1)
pos_ = vhas_.any(1) & ~vblr_
neg_ = ~vhas_.any(1) & ~vblr_
rows_ = []
for name, lo, hi in BUCKETS:
    m = (dval_ >= lo) & (dval_ <= hi)
    rows_.append({"bucket": name, "positives": int((m & pos_).sum()), "recall_v5": round(float(al_[m & pos_].mean()), 4) if (m & pos_).any() else None,
                  "recall_single_0.36": round(float(al1_[m & pos_].mean()), 4) if (m & pos_).any() else None,
                  "negatives": int((m & neg_).sum()), "negative_alarms_v5": round(float(al_[m & neg_].mean()), 4) if (m & neg_).any() else None})
# the same distances for the D-Fire TEST images (distribution only; no test score is computed here)
tp_, thas_, tblr_ = split_table("test")
tfp_ = np.stack([dhash16(cv2.imread(p)) for p in tp_])
dtest_ = hamming_min(tfp_, trfp_)
q = [10, 25, 50, 75, 90]
dist = {"thresholds": {"fire": TF, "smoke": TS}, "val_positives": int(pos_.sum()), "test_images": int(len(tp_)),
        "buckets_on_validation": rows_,
        "distance_to_nearest_training_image_percentiles": {"percentiles": q,
            "validation_positives": [int(v) for v in np.percentile(dval_[pos_], q)],
            "test_positives": [int(v) for v in np.percentile(dtest_[thas_.any(1)], q)]},
        "share_over_72_bits": {"validation_positives": round(float((dval_[pos_] > 72).mean()), 3),
                               "test_positives": round(float((dtest_[thas_.any(1)] > 72).mean()), 3)}}
report["val_distance"] = dist
json.dump(dist, open(f"{OUT}/val_distance.json", "w"), indent=1)
# the stems of validation images that are an unseen scene, for use as a stricter validation set later
json.dump(sorted(s for s, d in zip(vstem_, dval_) if d > 72), open(f"{OUT}/val_unseen_scene_stems.json", "w"))
print(json.dumps(dist, indent=1))


In [ ]:
# ---- Package and download ----
zp = f"{OUT}/{ZIP_NAME}"
with zipfile.ZipFile(zp + ".part", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in ("val_distance.json", "val_unseen_scene_stems.json"):
        z.write(f"{OUT}/{fn}", fn)
os.replace(zp + ".part", zp)
print("saved", zp)
try:
    from google.colab import files as _files
    _files.download(zp)
except Exception:  # noqa: BLE001
    print("Download it from Drive: folder", os.path.basename(OUT))
